In [ ]:
%pip install -r requirements.txt

# DATA STANDARDIZATION

In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
# Load RIS dataset
df = pd.read_csv(filepath_or_buffer='/path/document.csv', sep=';')

#Delete unneccessary columns & reorder dataframe
df = df.drop(columns=[])
new_order =['new order']
df_neworder = df.reindex(columns=new_order)

#Rename MRI devices with their respectiv magnetic flux density
df_devices = df_neworder.replace({'MRI device':'magnetic flux density'})

In [ ]:
# Function to extract paragraphs based on the starting keyword
def extract_paragraph(text, start_keywords):
    # Create a regex pattern that matches any of the start keywords
    start_pattern = '|'.join([re.escape(keyword) for keyword in start_keywords])
    pattern = re.compile(rf'({start_pattern})(.*?)(?=(\n[A-Z][^:]*:|\Z))', re.DOTALL)
    match = pattern.search(text)
    return match.group(2).strip() if match else None

# Keywords for clinical question & imaging procedure description
klinik_keywords = ['keywords']
technik_keywords = ['keywords']

# Apply the function to extract the required paragraphs and store in a new column
df_devices['new column clinical question'] = df_devices['report'].apply(lambda x: extract_paragraph(x, klinik_keywords))
df_devices['new column sequences'] = df_devices['report'].apply(lambda x: extract_paragraph(x, technik_keywords))

# Delete the originial report column
df_extracted = df_devices.drop(columns=['report'])

In [ ]:
# Keywords for contrast medium administration
keywords = ['keywords']

# Function to check if any keyword is in the text
def contains_keywords(text, keywords):
    return any(keyword in text for keyword in keywords)

# Create a new list to store the results
contrast_medium = df_extracted['sequences'].apply(lambda x: contains_keywords(x, keywords))

# Add the list as a new column to the DataFrame
df_extracted['new column contrast medium administration'] = contrast_medium

In [ ]:
#Save result in a new CSV file
df_extracted.to_csv('/path/standardized document.csv',sep=',')

# PROTOCOL PREDICTION OPEN SOURCE MODEL

In [ ]:
import pandas as pd
import replicate
import os
import openai
import re
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

## LLAMA 

### PIPELINE

In [ ]:
#Load data with clinical question
df = pd.read_csv(filepath_or_buffer='/path/clinical_questions.csv', sep=';')

#Load enviroment variables
load_dotenv()

#Set API Token
replicate_api_token = os.getenv('REPLICATE_API_TOKEN')
replicate_client = replicate.Client(api_token=replicate_api_token)

standard_sequences = ['Axiale T1','Coronare T1','Axiale T2','Sagittale T2','Axiale T2 FS','Coronare T2 FS','Axiale T2*','Axiale STIR','Sagittale STIR','Coronare TIR','Axiale T2 BLADE','Axiale T1 FLASH','Axiale T2 FLASH','Sagittale T1 FLASH','Coronare T1 FLASH','Axiale T2 SPACE','Sagittale T2 SPACE','Coronare FLAIR','Axiale FLAIR','Sagittale FLAIR','Coronare FLAIR FAST','3D FLAIR','3D T1 SPACE FS','3D T2 SPACE FS','T1 MPRAGE','T1 VIBE','Coronare T1 Dynamik','Axiale SWI','Axiale DWI','Coronare DWI','Axiale DTI','Art. TOF-MRA','ASL','Perfusion','Axiale T2 FS Hals','Axiale T1 FS Hals','Kopf-Hals-Angio','3D T1 SPACE FS Hals','fMRT','Spektroskopie','Axiale 3D T2','Coronare T2 FS','Coronare T1 FS','Axiale FGATIR']

#Prediction Pipeline
def prediction_prompt(row):
    input = {
    "top_p": 0.1,
    "prompt": (
        f"Du bist Neuroradiologe. Du bekommst eine radiologische Fragestellung.\n"
        f"Die Fragestellung enthält Abkürzungen. Formuliere dafür die Abkürzungen aus und antworte so: Abkürzung: ausgeschriebene Abkürzung.\n"
        f"Wenn es bereits eine Diagnose gibt, nenne diese. Führe dann die 3 wahrscheinlichsten Differentialdiagnosen für diese Fragestellung auf.\n"
        f"Dir stehen standardisierte MRT Sequenzen zur Verfügung: {standard_sequences}.\n"
        f"Nenne alle Sequenzen aus der Liste der standardisierten MRT Sequenzen, die in der MRT Untersuchung nötig sind, um die Fragestellung zu beantworten. Falls eine Sequenz vor und nach der Kontrastmittelgabe geplant werden soll, nenne sie zweifach.\n"
        f"Bestimme außerdem, ob in der MRT Untersuchung Kontrastmittel gegeben werden soll oder nicht. Wenn ja, schreibe TRUE, wenn nicht, schreibe FALSE.\n"
        f"Befolge stets das Antwortformat. Erkläre dein Vorgehen nicht. Gib nur die erfragten Informationen wieder, füge keine weiteren Zeichen hinzu.\n"
        f"Deine Antwort soll folgendes Format haben:\n"
        f"Abkürzungen: Abkürzung: ausgeschriebene Abkürzung, Abkürzung: ausgeschriebene Abkürzung, ...\n"
        f"Diagnose: bereits bekannte Diagnose, Differentialdiagnosen: 1. wahrscheinlichste, 2. zweitwahrscheinlichste, 3. drittwahrscheinlichste\n"
        f"Sequenzen: Sequenz 1, Sequenz 2, Sequenz 3, ...\n"
        f"Kontrastmittelgabe: TRUE oder FALSE\n"
        f"Radiologische Fragestellung: {row['Fragestellung']}"),
    "temperature": 0.1,
    "prompt_template": "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a neuroradiologist.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
    "presence_penalty": 1.15
}
    # Using Replicate's API to run the prediction
    output = replicate_client.run('meta/llama-4-maverick-instruct', input=input)
    
    # Join tokens into a coherent string
    if isinstance(output, list):
        output = ''.join(output)

    return output

# Iterate over the test DataFrame and generate predictions
results = []

for index, row in df.iterrows():  
    result = prediction_prompt(row)
    results.append(result)


### EXTRACT THE DATA OF THE RESULTS

In [ ]:
# Initialize the lists to store the extracted data
abbreviations = []
differential_diagnosis = []
sequence_prediction = []
contrastmedium_prediction = []

# Iterate through each entry in the results
for entry in results:
    # Match the 'Abkürzungen:' section
    abbreviation_match = re.search(r'Abkürzungen:\s*(.*?)\n', entry, re.DOTALL)
    if abbreviation_match:
        abbreviations.append(abbreviation_match.group(1).strip())
    else:
        abbreviations.append(None)  # Append None if the section is missing

    # Match the 'Differentialdiagnosen:' section
    dd_match = re.search(r'Diagnose:\s*(.*?)\n', entry, re.DOTALL)
    if dd_match:
        differential_diagnosis.append(dd_match.group(1).strip())
    else:
        differential_diagnosis.append(None)  # Append an None if the section is missing
    
    # Match the 'Sequenzen:' section
    sequenzen_match = re.search(r'Sequenzen:\s*(.*?)\n', entry, re.DOTALL)
    if (sequenzen_match):
        sequence_prediction.append(sequenzen_match.group(1).strip())
    else:
        sequence_prediction.append(None)  # Append None if the section is missing

    # Match the 'Kontrastmittelgabe:' section
    contrastmedium_match = re.search(r'Kontrastmittelgabe:\s*(TRUE|FALSE)', entry, re.DOTALL)
    if contrastmedium_match:
        contrastmedium_prediction.append(contrastmedium_match.group(1).strip())
    else:
        contrastmedium_prediction.append(None)  # Append None if the section is missing


#Add the extracted results to the dataframe
df.insert(8,'Abkürzungen',abbreviations)
df.insert(9,'Diagnose/DD',differential_diagnosis)
df.insert(10,'Vorhersage Sequenzen', sequence_prediction)
df.insert(11,'Vorhersage Kontrastmittelgabe', contrastmedium_prediction)

#Save the dataframe as csv
df.to_csv(path_or_buf='/path/results',sep=';')

## LLAMA WITH RAG

### PIPELINE

In [ ]:
# Load data with clinical question
df = pd.read_csv(filepath_or_buffer='/path/clinical_questions.csv', sep=';')

#Load enviroment variables
load_dotenv()

# Set API Token
replicate_api_token = os.getenv('REPLICATE_API_TOKEN')
replicate_client = replicate.Client(api_token=replicate_api_token)

# Set OpenAI API key
openai_api_key = os.getenv('OPENAI_API_KEY')
openai.api_key = openai_api_key

# Function to load and split PDF documents
def get_docs():
    loader_pdf = PyPDFLoader('/path/guidelines')
    pdf_doc = loader_pdf.load()

    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n \n", "\n", " ", ""],
        chunk_size=400,
        chunk_overlap=0,
        length_function=len,
        is_separator_regex=False
    )

    splitpdf = text_splitter.split_documents(pdf_doc)
    return splitpdf

# Function to create the vector store
def create_vector_store(docs):
    embedding = OpenAIEmbeddings(openai_api_key=openai_api_key, model="text-embedding-3-large")
    vectorStore = FAISS.from_documents(docs, embedding=embedding)
    return vectorStore

docs = get_docs()
vectorStore = create_vector_store(docs)


#Prediction Pipeline
def prediction_prompt(row):
    #Embed clinical question into query
    query = f"Das ist eine radiologische Fragestellung: {row['Fragestellung']}. Zu welcher 'Anwendung' gehört sie? Das benutzte 'Gerät' ist {row['Gerät']}."
    
    #Evoke retrieval of documents
    retriever = vectorStore.as_retriever(search_kwargs={"k": 8})
    retrieved_docs = retriever.get_relevant_documents(query)

    input = {
    "top_p": 0.1,
    "prompt": (
        f"Du bist Neuroradiologe. Du bekommst eine radiologische Fragestellung.\n"
        f"Die Fragestellung enthält Abkürzungen. Formuliere nur die Abkürzungen aus der Fragestellung aus und antworte so: Abkürzung: ausgeschriebene Abkürzung.\n"
        f"Wenn es bereits eine Diagnose gibt, nenne diese. Führe dann die 3 wahrscheinlichsten Differentialdiagnosen für diese Fragestellung auf.\n"
        f"Du hast MRT-Protokolle zur Verfügung: {retrieved_docs}. Jedes Protokoll enthält einen Abschnitt zur Anwendung. Dieser behinhaltet für welche radiologischen Fragestellungen das jeweilige Protokoll geeignet ist.\n"
        f"Wähle das Protokoll aus, was am besten zur Beantwortung dieser radiologischen Fragestellung passt. Berücksichtige hierbei vorallem die Details aus Anwendungsbeschreibung des Protokolls.\n"
        f"Gib die Informationen des Protokolls unverändert wieder. Befolge stets das Antwortformat. Erkläre dein Vorgehen nicht, füge keine weiteren Zeichen hinzu.\n"
        f"Deine Antwort soll folgendes Format haben:\n"
        f"Abkürzungen: Abkürzung: ausgeschriebene Abkürzung, Abkürzung: ausgeschriebene Abkürzung, ...\n"
        f"Diagnose: bereits bekannte Diagnose, Differentialdiagnosen: 1. wahrscheinlichste, 2. zweitwahrscheinlichste, 3. drittwahrscheinlichste\n"
        f"Protokollname: Name des Protokolls\n"
        f"Sequenzen: Sequenzen des Protokolls\n"
        f"Kontrastmittelgabe: Kontrastmittelgabe im Protokoll\n"
        f"Radiologische Fragestellung: {row['Fragestellung']}"),
    "temperature": 0.1,
    "prompt_template": "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a neuroradiologist.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
    "presence_penalty": 1.15
}
    # Using Replicate's API to run the prediction
    output = replicate_client.run('meta/llama-4-maverick-instruct', input=input)
    
    # Join tokens into a coherent string
    if isinstance(output, list):
        output = ''.join(output)

    return output, retrieved_docs

# Iterate over the test DataFrame and generate predictions
results = []
retrieved_documents = []

for index, row in df.iterrows():  
    result, retrieved_docs = prediction_prompt(row)
    results.append(result)
    retrieved_documents.append(retrieved_docs)



### EXTRACT THE DATA OF THE RESULTS

In [ ]:
retrieved_docs_name = []

# Iterate through each list in the retrieved_documents and extract protocol name
for entry_list in retrieved_documents:
    name_group = []

    if isinstance(entry_list, list):
        # Iterate through each document-like object in the list
        for entry in entry_list:
            # Check if the entry has 'page_content'
            if hasattr(entry, 'page_content'):
                page_content = entry.page_content
                
                # Split the content by lines
                lines = page_content.split('\n')
                
                for line in lines:
                    # Adjust to match both 'Name :' and 'Name:'
                    if re.match(r"Name\s*:", line.strip()):
                        # Extract the part after 'Name:'
                        name = line.split(":", 1)[1].strip()
                        name_group.append(name)

    # Append the group of names (per row) to the main list
    retrieved_docs_name.append(name_group)



In [ ]:
# Initialize the lists to store the extracted data
abbreviations = []
differential_diagnosis = []
protocol_prediction = []
sequence_prediction = []
contrastmedium_prediction = []

# Iterate through each entry in the results
for entry in results:
    # Match the 'Abkürzungen:' section
    abbreviation_match = re.search(r'Abkürzungen:\s*(.*?)\n', entry, re.DOTALL)
    if abbreviation_match:
        abbreviations.append(abbreviation_match.group(1).strip())
    else:
        abbreviations.append(None)  # Append None if the section is missing

    # Match the 'Differentialdiagnosen:' section
    dd_match = re.search(r'Diagnose:\s*(.*?)\n', entry, re.DOTALL)
    if dd_match:
        differential_diagnosis.append(dd_match.group(1).strip())
    else:
        differential_diagnosis.append(None)  # Append an None if the section is missing
    
    # Match the 'Protokollname:' section
    protocol_match = re.search(r'Protokollname:\s*(.*?)\n', entry, re.DOTALL)
    if protocol_match:
        protocol_prediction.append(protocol_match.group(1).strip())
    else:
        protocol_prediction.append(None)  # Append None if the section is missing

    # Match the 'Sequenzen:' section
    sequenzen_match = re.search(r'Sequenzen:\s*(.*?)\n', entry, re.DOTALL)
    if sequenzen_match:
        sequence_prediction.append(sequenzen_match.group(1).strip())
    else:
        sequence_prediction.append(None)  # Append None if the section is missing

    # Match the 'Kontrastmittelgabe:' section
    contrastmedium_match = re.search(r'Kontrastmittelgabe:\s*(ja|nein|gegebenenfalls)', entry, re.DOTALL)
    if contrastmedium_match:
        contrastmedium_prediction.append(contrastmedium_match.group(1).strip())
    else:
        contrastmedium_prediction.append(None)  # Append None if the section is missing

#Add extrated results to dataframe 
df.insert(7,'Abkürzungen',abbreviations)
df.insert(8,'Diagnose/DD',differential_diagnosis)
df.insert(9,'Vorhersage Protokollname',protocol_prediction)
df.insert(10,'Vorhersage Sequenzen', sequence_prediction)
df.insert(11,'Vorhersage Kontrastmittelgabe', contrastmedium_prediction)
df.insert(12,'Retrieved Documents',retrieved_docs_name)

#Save the dataframe as CSV
df.to_csv(path_or_buf='/path/results',sep=';')

# PROTOCOL PREDICTION PROPRIETARY MODELS

In [ ]:
import pandas as pd
import replicate
import os
import openai
import re
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

## GPT

### PIPELINE

In [ ]:
# Load environment variables
load_dotenv()

# Set OpenAI API key
openai_api_key = os.getenv('OPENAI_API_KEY')
openai.api_key = openai_api_key

# Load the data with clinical questions
df = pd.read_csv(filepath_or_buffer='/path/clinical_questions.csv', sep=';')

standard_sequences = ['Axiale T1','Coronare T1','Axiale T2','Sagittale T2','Axiale T2 FS','Coronare T2 FS','Axiale T2*','Axiale STIR','Sagittale STIR','Coronare TIR','Axiale T2 BLADE','Axiale T1 FLASH','Axiale T2 FLASH','Sagittale T1 FLASH','Coronare T1 FLASH','Axiale T2 SPACE','Sagittale T2 SPACE','Coronare FLAIR','Axiale FLAIR','Sagittale FLAIR','Coronare FLAIR FAST','3D FLAIR','3D T1 SPACE FS','3D T2 SPACE FS','T1 MPRAGE','T1 VIBE','Coronare T1 Dynamik','Axiale SWI','Axiale DWI','Coronare DWI','Axiale DTI','Art. TOF-MRA','ASL','Perfusion','Axiale T2 FS Hals','Axiale T1 FS Hals','Kopf-Hals-Angio','3D T1 SPACE FS Hals','fMRT','Spektroskopie','Axiale 3D T2','Coronare T2 FS','Coronare T1 FS','Axiale FGATIR']

def prediction_prompt(standard_sequences,row):
    prompt = (
        f"Du bist Neuroradiologe. Du bekommst eine radiologische Fragestellung.\n"
        f"Die Fragestellung enthält Abkürzungen. Formuliere dafür die Abkürzungen aus und antworte so: Abkürzung: ausgeschriebene Abkürzung.\n"
        f"Wenn es bereits eine Diagnose gibt, nenne diese. Führe dann die 3 wahrscheinlichsten Differentialdiagnosen für diese Fragestellung auf.\n"
        f"Dir stehen standardisierte MRT Sequenzen zur Verfügung: {standard_sequences}.\n"
        f"Nenne alle Sequenzen aus der Liste der standardisierten MRT Sequenzen, die in der MRT Untersuchung nötig sind, um die Fragestellung zu beantworten. Falls eine Sequenz vor und nach der Kontrastmittelgabe geplant werden soll, nenne sie zweifach.\n"
        f"Bestimme außerdem, ob in der MRT Untersuchung Kontrastmittel gegeben werden soll oder nicht. Wenn ja, schreibe TRUE, wenn nicht, schreibe FALSE.\n"
        f"Befolge stets das Antwortformat. Erkläre dein Vorgehen nicht. Gib nur die erfragten Informationen wieder, füge keine weiteren Zeichen hinzu.\n"
        f"Deine Antwort soll folgendes Format haben:\n"
        f"Abkürzungen: Abkürzung: ausgeschriebene Abkürzung, Abkürzung: ausgeschriebene Abkürzung, ...\n"
        f"Diagnose: bereits bekannte Diagnose, Differentialdiagnosen: 1. wahrscheinlichste, 2. zweitwahrscheinlichste, 3. drittwahrscheinlichste\n"
        f"Sequenzen: Sequenz 1, Sequenz 2, Sequenz 3, ...\n"
        f"Kontrastmittelgabe: TRUE oder FALSE\n"
        f"Radiologische Fragestellung: {row['Fragestellung']}"
    )
    completion = openai.chat.completions.create(
        model="gpt-5.2-2025-12-11",
        temperature=0.1,
        top_p=0.1,
        presence_penalty=1.15,
        messages=[
            {"role": "system", "content": "You are a neuroradiologist."},
            {"role": "user", "content": prompt}
        ]
    )
    
    result = completion.choices[0].message.content.strip()

    print(result)
    return result

# Iterate over the test DataFrame and generate predictions
results = []

for index, row in df.iterrows():
    result = prediction_prompt(standard_sequences,row)
    results.append(result)

print(results)
    

### DATA EXTRACTION OUTPUT

In [ ]:
# Initialize the lists to store the extracted data
abbreviations = []
differential_diagnosis = []
sequence_prediction = []
contrastmedium_prediction= []

# Iterate through each entry in the results
for entry in results:
    # Match the 'Abkürzungen:' section
    abbreviation_match = re.search(r'Abkürzungen:\s*(.*?)\n', entry, re.DOTALL)
    if abbreviation_match:
        abbreviations.append(abbreviation_match.group(1).strip())

    # Match the 'Differentialdiagnosen:' section
    dd_match = re.search(r'Diagnose:\s*(.*?)\n', entry, re.DOTALL)
    if dd_match:
        # Extract the list of differential diagnoses
        dd_list = [dd.strip() for dd in dd_match.group(1).split(',')]
        differential_diagnosis.append(dd_list)

    # Match the 'Sequenzen:' section
    sequenzen_match = re.search(r'Sequenzen:\s*(.*?)\n', entry, re.DOTALL)
    if sequenzen_match:
        sequence_prediction.append(sequenzen_match.group(1).strip())
    

    # Match the 'Kontrastmittelgabe:' section
    contrastmedium_match = re.search(r'Kontrastmittelgabe:\s*(TRUE|FALSE)', entry, re.DOTALL)
    if contrastmedium_match:
        contrastmedium_prediction.append(contrastmedium_match.group(1).strip())

#Add extracted results to the dataframe
df.insert(8,'Abkürzungen',abbreviations)
df.insert(9,'Diagnose/DD',differential_diagnosis)
df.insert(10,'Vorhersage Sequenzen', sequence_prediction)
df.insert(11,'Vorhersage Kontrastmittelgabe', contrastmedium_prediction)

#Save as csv
df.to_csv(path_or_buf='/path/results',sep=';')

## GPT WITH RAG

### PIPELINE

In [ ]:
# Load environment variables
load_dotenv()

# Set OpenAI API key
openai_api_key = os.getenv('OPENAI_API_KEY')
openai.api_key = openai_api_key

# Load the CSV file
df = pd.read_csv(filepath_or_buffer='/path/clinical_questions.csv', sep=';')

# Function to load and split the PDF document
def get_docs():
    loader_pdf = PyPDFLoader('/path/guidelines')
    pdf_doc = loader_pdf.load()
    
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n \n", "\n", " ", ""],
        chunk_size=450,
        chunk_overlap=0,
        length_function=len,
        is_separator_regex=False
    )

    splitpdf = text_splitter.split_documents(pdf_doc)
    return splitpdf

# Function to create the vector store
def create_vector_store(docs):
    embedding = OpenAIEmbeddings(openai_api_key=openai_api_key, model="text-embedding-3-large")
    vectorStore = FAISS.from_documents(docs, embedding=embedding)
    return vectorStore

docs = get_docs()
vectorStore = create_vector_store(docs)

# Prediction Pipeline
def prediction_prompt(row):
    #Embed clinical question into query
    query = f"Das ist eine radiologische Fragestellung:{row['Fragestellung']}. Zu welcher 'Anwendung' gehört sie? Das benutzte 'Gerät' ist {row['Gerät']}."

    #Evoke retrieval of documents
    retriever = vectorStore.as_retriever(search_kwargs={"k": 8})
    retrieved_docs = retriever.get_relevant_documents(query)

    prompt = (
        f"Du bist Neuroradiologe. Du bekommst eine radiologische Fragestellung.\n"
        f"Die Fragestellung enthält Abkürzungen. Formuliere nur die Abkürzungen aus der Fragestellung aus und antworte so: Abkürzung: ausgeschriebene Abkürzung.\n"
        f"Wenn es bereits eine Diagnose gibt, nenne diese. Führe dann die 3 wahrscheinlichsten Differentialdiagnosen für diese Fragestellung auf.\n"
        f"Du hast MRT-Protokolle zur Verfügung: {retrieved_docs}. Jedes Protokoll enthält einen Abschnitt zur Anwendung. Dieser behinhaltet für welche radiologischen Fragestellungen das jeweilige Protokoll geeignet ist.\n"
        f"Wähle das Protokoll aus, was am besten zur Beantwortung dieser radiologischen Fragestellung passt. Berücksichtige hierbei vorallem die Details aus Anwendungsbeschreibung des Protokolls.\n"
        f"Gib die Informationen des Protokolls unverändert wieder. Befolge stets das Antwortformat. Erkläre dein Vorgehen nicht, füge keine weiteren Zeichen hinzu.\n"
        f"Deine Antwort soll folgendes Format haben:\n"
        f"Abkürzungen: Abkürzung: ausgeschriebene Abkürzung, Abkürzung: ausgeschriebene Abkürzung, ...\n"
        f"Diagnose: bereits bekannte Diagnose, Differentialdiagnosen: 1. wahrscheinlichste, 2. zweitwahrscheinlichste, 3. drittwahrscheinlichste\n"
        f"Protokollname: Name des Protokolls\n"
        f"Sequenzen: Sequenzen des Protokolls\n"
        f"Kontrastmittelgabe: Kontrastmittelgabe im Protokoll\n"
        f"Radiologische Fragestellung: {row['Fragestellung']}"
        )
    
    completion = openai.chat.completions.create(
        model="gpt-5.2-2025-12-11",
        temperature=0.1,
        top_p=0.1,
        presence_penalty=1.15,
        messages=[
            {"role": "system", "content": "You are a neuroradiologist."},
            {"role": "user", "content": prompt}
        ]
    )
    
    result = completion.choices[0].message.content.strip()

    return result, retrieved_docs

# Iterate over the test DataFrame and generate predictions
results = []
retrieved_documents = []

for index, row in df.iterrows():  
    result, retrieved_docs = prediction_prompt(row)
    results.append(result)
    retrieved_documents.append(retrieved_docs)

### DATA EXTRACTION OUTPUT

In [ ]:
retrieved_docs_name = []

# Iterate through each list in the retrieved_documents and extract protocol name
for entry_list in retrieved_documents:
    name_group = []  

    if isinstance(entry_list, list):
        # Iterate through each document-like object in the list
        for entry in entry_list:
            # Check if the entry has 'page_content'
            if hasattr(entry, 'page_content'):
                page_content = entry.page_content
                
                # Split the content by lines
                lines = page_content.split('\n')
                
                for line in lines:
                    if re.match(r"Name\s*:", line.strip()):
                        # Extract the part after 'Name:'
                        name = line.split(":", 1)[1].strip()
                        name_group.append(name)

    # Append the group of names (per row) to the main list
    retrieved_docs_name.append(name_group)

In [ ]:
# Initialize the lists to store the extracted data
abbreviations = []
diagnosis = []
differential_diagnosis = []
protocol_prediction = []
sequence_prediction = []
contrastmedium_prediction = []

# Iterate through each entry in the results
for entry in results:
    # Match the 'Abkürzungen:' section
    abbreviation_match = re.search(r'Abkürzungen:\s*(.*?)\n(?:Diagnose|Differentialdiagnosen):', entry, re.DOTALL)
    if abbreviation_match:
        abbreviations.append(abbreviation_match.group(1).strip())

    # Match the 'Diagnose:' section
    diagnosis_match = re.search(r'Diagnose:\s*(.*?)\n', entry, re.DOTALL)
    if diagnosis_match:
        diagnosis.append(diagnosis_match.group(1).strip())

    # Match the 'Differentialdiagnosen:' section
    dd_match = re.search(r'Diagnose:\s*(.*?)\n', entry, re.DOTALL)
    if dd_match:
        differential_diagnosis.append(dd_match.group(1).strip())
    else:
        differential_diagnosis.append(None)  # Append an None if the section is missing

    # Match the 'Protokollname:' section
    protocol_match = re.search(r'Protokollname:\s*(.*?)\n', entry, re.DOTALL)
    if protocol_match:
        protocol_prediction.append(protocol_match.group(1).strip())
        
    # Match the 'Sequenzen:' section
    #sequenzen_match = re.search(r'Sequenzen:\s*(.*?)\n(?:Kontrastmittelgabe):', entry, re.DOTALL)
    #if sequenzen_match:
        #sequence_prediction.append(sequenzen_match.group(1).strip())
     # Match the 'Sequenzen:' section
    
    sequenzen_match = re.search(r'Sequenzen:\s*(.*?)\n(?:Kontrastmittelgabe):', entry, re.DOTALL)
    if sequenzen_match:
        # Tabs entfernen + (optional) mehrfach Spaces normalisieren
        seq = sequenzen_match.group(1).replace('\t', ' ')
        seq = re.sub(r' +', ' ', seq).strip()
        sequence_prediction.append(seq)

    # Match the 'Kontrastmittelgabe:' section
    contrastmedium_match = re.search(r'Kontrastmittelgabe:\s*(\S+)', entry)
    if contrastmedium_match:
        contrastmedium_prediction.append(contrastmedium_match.group(1).strip())

#Add extracted results to the dataframe
df.insert(7,'Abkürzungen', abbreviations)
df.insert(8,'Diagnose', diagnosis)
df.insert(9,'Diagnosen', differential_diagnosis)
df.insert(10,'Vorhersage Protokollname',protocol_prediction)
df.insert(11,'Vorhersage Sequenzen', sequence_prediction)
df.insert(12,'Vorhersage Kontrastmittelgabe', contrastmedium_prediction)
df.insert(13,'Retrieved Documents',retrieved_docs_name)


#Save as csv
df.to_csv(path_or_buf='path/results.csv',sep=';')

## CLAUDE

### PIPELINE

In [ ]:
#Load data with clinical question
df = pd.read_csv(filepath_or_buffer='/path/clinical_questions.csv', sep=';')

#Load enviroment variables
load_dotenv()

#Set API Token
replicate_api_token = os.getenv('REPLICATE_API_TOKEN')
replicate_client = replicate.Client(api_token=replicate_api_token)

standard_sequences = ['Axiale T1','Coronare T1','Axiale T2','Sagittale T2','Axiale T2 FS','Coronare T2 FS','Axiale T2*','Axiale STIR','Sagittale STIR','Coronare TIR','Axiale T2 BLADE','Axiale T1 FLASH','Axiale T2 FLASH','Sagittale T1 FLASH','Coronare T1 FLASH','Axiale T2 SPACE','Sagittale T2 SPACE','Coronare FLAIR','Axiale FLAIR','Sagittale FLAIR','Coronare FLAIR FAST','3D FLAIR','3D T1 SPACE FS','3D T2 SPACE FS','T1 MPRAGE','T1 VIBE','Coronare T1 Dynamik','Axiale SWI','Axiale DWI','Coronare DWI','Axiale DTI','Art. TOF-MRA','ASL','Perfusion','Axiale T2 FS Hals','Axiale T1 FS Hals','Kopf-Hals-Angio','3D T1 SPACE FS Hals','fMRT','Spektroskopie','Axiale 3D T2','Coronare T2 FS','Coronare T1 FS','Axiale FGATIR']

#Prediction Pipeline
def prediction_prompt(row):
    input = {
    "top_p": 0.1,
    "prompt": (
        f"Du bist Neuroradiologe. Du bekommst eine radiologische Fragestellung.\n"
        f"Die Fragestellung enthält Abkürzungen. Formuliere dafür die Abkürzungen aus und antworte so: Abkürzung: ausgeschriebene Abkürzung.\n"
        f"Wenn es bereits eine Diagnose gibt, nenne diese. Führe dann die 3 wahrscheinlichsten Differentialdiagnosen für diese Fragestellung auf.\n"
        f"Dir stehen standardisierte MRT Sequenzen zur Verfügung: {standard_sequences}.\n"
        f"Nenne alle Sequenzen aus der Liste der standardisierten MRT Sequenzen, die in der MRT Untersuchung nötig sind, um die Fragestellung zu beantworten. Falls eine Sequenz vor und nach der Kontrastmittelgabe geplant werden soll, nenne sie zweifach.\n"
        f"Bestimme außerdem, ob in der MRT Untersuchung Kontrastmittel gegeben werden soll oder nicht. Wenn ja, schreibe TRUE, wenn nicht, schreibe FALSE.\n"
        f"Befolge stets das Antwortformat. Erkläre dein Vorgehen nicht. Gib nur die erfragten Informationen wieder, füge keine weiteren Zeichen hinzu.\n"
        f"Deine Antwort soll folgendes Format haben:\n"
        f"Abkürzungen: Abkürzung: ausgeschriebene Abkürzung, Abkürzung: ausgeschriebene Abkürzung, ...\n"
        f"Diagnose: bereits bekannte Diagnose, Differentialdiagnosen: 1. wahrscheinlichste, 2. zweitwahrscheinlichste, 3. drittwahrscheinlichste\n"
        f"Sequenzen: Sequenz 1, Sequenz 2, Sequenz 3, ...\n"
        f"Kontrastmittelgabe: TRUE oder FALSE\n"
        f"Radiologische Fragestellung: {row['Fragestellung']}"),
    "temperature": 0.1,
    "prompt_template": "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a neuroradiologist.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
    "presence_penalty": 1.15
}
    # Using Replicate's API to run the prediction
    output = replicate_client.run('anthropic/claude-opus-4.6', input=input)
    
    # Join tokens into a coherent string
    if isinstance(output, list):
        output = ''.join(output)

    return output

# Iterate over the test DataFrame and generate predictions
results = []

for index, row in df.iterrows():  
    result = prediction_prompt(row)
    results.append(result)


### DATA EXTRACTION OUTPUT

In [ ]:
# Initialize the lists to store the extracted data
abbreviations = []
differential_diagnosis = []
sequence_prediction = []
contrastmedium_prediction = []

# Iterate through each entry in the results
for entry in results:
    # Match the 'Abkürzungen:' section
    abbreviation_match = re.search(r'Abkürzungen:\s*(.*?)\n', entry, re.DOTALL)
    if abbreviation_match:
        abbreviations.append(abbreviation_match.group(1).strip())
    else:
        abbreviations.append(None)  # Append None if the section is missing

    # Match the 'Differentialdiagnosen:' section
    dd_match = re.search(r'Diagnose:\s*(.*?)\n', entry, re.DOTALL)
    if dd_match:
        differential_diagnosis.append(dd_match.group(1).strip())
    else:
        differential_diagnosis.append(None)  # Append an None if the section is missing
    
    # Match the 'Sequenzen:' section
    sequenzen_match = re.search(r'Sequenzen:\s*(.*?)\n', entry, re.DOTALL)
    if (sequenzen_match):
        sequence_prediction.append(sequenzen_match.group(1).strip())
    else:
        sequence_prediction.append(None)  # Append None if the section is missing

    # Match the 'Kontrastmittelgabe:' section
    contrastmedium_match = re.search(r'Kontrastmittelgabe:\s*(TRUE|FALSE)', entry, re.DOTALL)
    if contrastmedium_match:
        contrastmedium_prediction.append(contrastmedium_match.group(1).strip())
    else:
        contrastmedium_prediction.append(None)  # Append None if the section is missing


#Add the extracted results to the dataframe
df.insert(8,'Abkürzungen',abbreviations)
df.insert(9,'Diagnose/DD',differential_diagnosis)
df.insert(10,'Vorhersage Sequenzen', sequence_prediction)
df.insert(11,'Vorhersage Kontrastmittelgabe', contrastmedium_prediction)

#Save the dataframe as csv
df.to_csv(path_or_buf='/path/results.csv',sep=';')

## CLAUDE WITH RAG

### PIPELINE

In [ ]:
# Load data with clinical question
df = pd.read_csv(filepath_or_buffer='/path/clinical_questions.csv', sep=';')

#Load enviroment variables
load_dotenv()

# Set API Token
replicate_api_token = os.getenv('REPLICATE_API_TOKEN')
replicate_client = replicate.Client(api_token=replicate_api_token)

# Set OpenAI API key
openai_api_key = os.getenv('OPENAI_API_KEY')
openai.api_key = openai_api_key

# Function to load and split PDF documents
def get_docs():
    loader_pdf = PyPDFLoader('/path/guidelines')
    pdf_doc = loader_pdf.load()

    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\n \n", "\n", " ", ""],
        chunk_size=400,
        chunk_overlap=0,
        length_function=len,
        is_separator_regex=False
    )

    splitpdf = text_splitter.split_documents(pdf_doc)
    return splitpdf

# Function to create the vector store
def create_vector_store(docs):
    embedding = OpenAIEmbeddings(openai_api_key=openai_api_key, model="text-embedding-3-large")
    vectorStore = FAISS.from_documents(docs, embedding=embedding)
    return vectorStore

docs = get_docs()
vectorStore = create_vector_store(docs)


#Prediction Pipeline
def prediction_prompt(row):
    #Embed clinical question into query
    query = f"Das ist eine radiologische Fragestellung: {row['Fragestellung']}. Zu welcher 'Anwendung' gehört sie? Das benutzte 'Gerät' ist {row['Gerät']}."
    
    #Evoke retrieval of documents
    retriever = vectorStore.as_retriever(search_kwargs={"k": 8})
    retrieved_docs = retriever.get_relevant_documents(query)

    input = {
    "top_p": 0.1,
    "prompt": (
        f"Du bist Neuroradiologe. Du bekommst eine radiologische Fragestellung.\n"
        f"Die Fragestellung enthält Abkürzungen. Formuliere nur die Abkürzungen aus der Fragestellung aus und antworte so: Abkürzung: ausgeschriebene Abkürzung.\n"
        f"Wenn es bereits eine Diagnose gibt, nenne diese. Führe dann die 3 wahrscheinlichsten Differentialdiagnosen für diese Fragestellung auf.\n"
        f"Du hast MRT-Protokolle zur Verfügung: {retrieved_docs}. Jedes Protokoll enthält einen Abschnitt zur Anwendung. Dieser behinhaltet für welche radiologischen Fragestellungen das jeweilige Protokoll geeignet ist.\n"
        f"Wähle das Protokoll aus, was am besten zur Beantwortung dieser radiologischen Fragestellung passt. Berücksichtige hierbei vorallem die Details aus Anwendungsbeschreibung des Protokolls.\n"
        f"Gib die Informationen des Protokolls unverändert wieder. Befolge stets das Antwortformat. Erkläre dein Vorgehen nicht, füge keine weiteren Zeichen hinzu.\n"
        f"Deine Antwort soll folgendes Format haben:\n"
        f"Abkürzungen: Abkürzung: ausgeschriebene Abkürzung, Abkürzung: ausgeschriebene Abkürzung, ...\n"
        f"Diagnose: bereits bekannte Diagnose, Differentialdiagnosen: 1. wahrscheinlichste, 2. zweitwahrscheinlichste, 3. drittwahrscheinlichste\n"
        f"Protokollname: Name des Protokolls\n"
        f"Sequenzen: Sequenzen des Protokolls\n"
        f"Kontrastmittelgabe: Kontrastmittelgabe im Protokoll\n"
        f"Radiologische Fragestellung: {row['Fragestellung']}"),
    "temperature": 0.1,
    "prompt_template": "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a neuroradiologist.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n",
    "presence_penalty": 1.15
}
    # Using Replicate's API to run the prediction
    output = replicate_client.run('anthropic/claude-opus-4.6', input=input)
    
    # Join tokens into a coherent string
    if isinstance(output, list):
        output = ''.join(output)

    return output, retrieved_docs

# Iterate over the test DataFrame and generate predictions
results = []
retrieved_documents = []

for index, row in df.iterrows():  
    result, retrieved_docs = prediction_prompt(row)
    results.append(result)
    retrieved_documents.append(retrieved_docs)



### DATA EXTRACTION OUTPUT

In [ ]:
retrieved_docs_name = []

# Iterate through each list in the retrieved_documents and extract protocol name
for entry_list in retrieved_documents:
    name_group = []

    if isinstance(entry_list, list):
        # Iterate through each document-like object in the list
        for entry in entry_list:
            # Check if the entry has 'page_content'
            if hasattr(entry, 'page_content'):
                page_content = entry.page_content
                
                # Split the content by lines
                lines = page_content.split('\n')
                
                for line in lines:
                    # Adjust to match both 'Name :' and 'Name:'
                    if re.match(r"Name\s*:", line.strip()):
                        # Extract the part after 'Name:'
                        name = line.split(":", 1)[1].strip()
                        name_group.append(name)

    # Append the group of names (per row) to the main list
    retrieved_docs_name.append(name_group)



In [ ]:
# Initialize the lists to store the extracted data
abbreviations = []
differential_diagnosis = []
protocol_prediction = []
sequence_prediction = []
contrastmedium_prediction = []

# Iterate through each entry in the results
for entry in results:
    # Match the 'Abkürzungen:' section
    abbreviation_match = re.search(r'Abkürzungen:\s*(.*?)\n', entry, re.DOTALL)
    if abbreviation_match:
        abbreviations.append(abbreviation_match.group(1).strip())
    else:
        abbreviations.append(None)  # Append None if the section is missing

    # Match the 'Differentialdiagnosen:' section
    dd_match = re.search(r'Diagnose:\s*(.*?)\n', entry, re.DOTALL)
    if dd_match:
        differential_diagnosis.append(dd_match.group(1).strip())
    else:
        differential_diagnosis.append(None)  # Append an None if the section is missing
    
    # Match the 'Protokollname:' section
    protocol_match = re.search(r'Protokollname:\s*(.*?)\n', entry, re.DOTALL)
    if protocol_match:
        protocol_prediction.append(protocol_match.group(1).strip())
    else:
        protocol_prediction.append(None)  # Append None if the section is missing

    # Match the 'Sequenzen:' section
    sequenzen_match = re.search(r'Sequenzen:\s*(.*?)\n', entry, re.DOTALL)
    if sequenzen_match:
        sequence_prediction.append(sequenzen_match.group(1).strip())
    else:
        sequence_prediction.append(None)  # Append None if the section is missing

    # Match the 'Kontrastmittelgabe:' section
    contrastmedium_match = re.search(r'Kontrastmittelgabe:\s*(ja|nein|gegebenenfalls)', entry, re.DOTALL)
    if contrastmedium_match:
        contrastmedium_prediction.append(contrastmedium_match.group(1).strip())
    else:
        contrastmedium_prediction.append(None)  # Append None if the section is missing

#Add extrated results to dataframe 
df.insert(7,'Abkürzungen',abbreviations)
df.insert(8,'Diagnose/DD',differential_diagnosis)
df.insert(9,'Vorhersage Protokollname',protocol_prediction)
df.insert(10,'Vorhersage Sequenzen', sequence_prediction)
df.insert(11,'Vorhersage Kontrastmittelgabe', contrastmedium_prediction)
df.insert(12,'Retrieved Documents',retrieved_docs_name)

#Save the dataframe as CSV
df.to_csv(path_or_buf='/path/results.csv',sep=';')

# STATISTICAL ANALYSIS

In [ ]:
from collections import Counter
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon
import numpy as np
import pandas as pd
import re 
import ast

## MODEL ACCURACIES

In [ ]:
#Load results
df_OS_woRAG = pd.read_csv(filepath_or_buffer='/path/results',sep=';')
df_OS_RAG = pd.read_csv(filepath_or_buffer='/path/results',sep=';')
df_woRAG = pd.read_csv(filepath_or_buffer='/path/results',sep=';')
df_RAG = pd.read_csv(filepath_or_buffer='/path/results',sep=';')
df_claude_woRAG = pd.read_csv(filepath_or_buffer='/path/results',sep=';')
df_claude_RAG =pd.read_csv(filepath_or_buffer='/path/results',sep=';')

#Import Ground Truth
df_GT = pd.read_csv(filepath_or_buffer='path/ground truth', sep=';')

### LLAMA

In [ ]:
#Tokenize sequences
def tokenize(text):
    tokens = [token.strip() for token in text.split(',')]
    return tokens

#Function for calculation of token-based accuracy
def calculate_symmetric_token_based_accuracy(ground_truth, result):
    ground_truth_tokens = tokenize(ground_truth)
    result_tokens = tokenize(result)
    
    ground_truth_counts = Counter(ground_truth_tokens)
    result_counts = Counter(result_tokens)
    
    # Calculate the number of matching tokens considering repetitions
    matching_tokens_gt_to_res = sum(min(ground_truth_counts[token], result_counts[token]) for token in ground_truth_counts)
    matching_tokens_res_to_gt = sum(min(ground_truth_counts[token], result_counts[token]) for token in result_counts)
    
    # Total tokens in ground truth and result (considering repetitions)
    total_tokens_gt = sum(ground_truth_counts.values())
    total_tokens_res = sum(result_counts.values())
    
    # Calculate accuracy from ground truth to result
    accuracy_gt_to_res = matching_tokens_gt_to_res / total_tokens_gt if total_tokens_gt > 0 else 0
    
    # Calculate accuracy from result to ground truth
    accuracy_res_to_gt = matching_tokens_res_to_gt / total_tokens_res if total_tokens_res > 0 else 0
    
    # Symmetric accuracy: average of both directions
    symmetric_accuracy = (accuracy_gt_to_res + accuracy_res_to_gt) / 2
    
    return symmetric_accuracy

# Provide the data
ground_truths = df_GT['Sequenzen'] 
results1 = df_OS_woRAG['Vorhersage Sequenzen']

accuracies_OS_woRAG = []
for gt, res in zip(ground_truths, results1):
    accuracy = calculate_symmetric_token_based_accuracy(gt, res)
    accuracies_OS_woRAG.append(accuracy)

#Calculate average accuracy
average_accuracy_seq = sum(accuracies_OS_woRAG) / len(accuracies_OS_woRAG)

# Bootstrap function to calculate confidence intervals
def bootstrap_confidence_interval(data, num_samples=10000, confidence_level=0.95):
    # Resample with replacement and calculate means
    sample_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(num_samples)]

    # Calculate the percentiles for the given confidence level
    lower_bound = np.percentile(sample_means, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(sample_means, (1 + confidence_level) / 2 * 100)

    return lower_bound, upper_bound
confidence_interval_seq = bootstrap_confidence_interval(accuracies_OS_woRAG)

#Contrast medium administration
#Replace values and change dtype
df_OS_woRAG['Vorhersage Kontrastmittelgabe'] = df_OS_woRAG['Vorhersage Kontrastmittelgabe'].replace({'ja': 1,'nein':0,'gegebenenfalls':0})
df_OS_woRAG['Vorhersage Kontrastmittelgabe'] = df_OS_woRAG['Vorhersage Kontrastmittelgabe'].astype(bool)

# Comparing the two columns and calculating accuracy
df_equal_OS_woRAG = df_OS_woRAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 

# Calculating the number of correct predictions (where the comparison is True)
correct_predictions = df_equal_OS_woRAG.sum()

# Calculating accuracy
average_accuracy_cm = correct_predictions / len(df_equal_OS_woRAG) * 100

# Set the number of bootstrap samples
n_iterations = 10000
bootstrap_accuracies = []

# Number of total samples
n = len(df_equal_OS_woRAG)

# Perform bootstrapping
for i in range(n_iterations):
    # Sample with replacement
    bootstrap_sample = np.random.choice(df_equal_OS_woRAG, size=n, replace=True)
    
    # Calculate accuracy for this bootstrap sample
    accuracies = np.sum(bootstrap_sample) / len(bootstrap_sample) * 100
    bootstrap_accuracies.append(accuracies)

# Convert to a numpy array for easier manipulation
bootstrap_accuracies = np.array(bootstrap_accuracies)

# Calculate the 95% confidence interval
confidence_interval_cm = np.percentile(bootstrap_accuracies, [2.5, 97.5])

data_results = [{
    'Confidenz Interval Sequences': (confidence_interval_seq[0] * 100, confidence_interval_seq[1] * 100),
    'Average Accuracy Sequences': average_accuracy_seq * 100,
    'Kappa Score Sequences':'-',
    'Confidenz Interval Contrastmedium': (confidence_interval_cm[0], confidence_interval_cm[1]),
    'Average Accuracy Contrastmedium': average_accuracy_cm,
    'Kappa Score Contrastmedium':'-',
    'Number of Correct Retrieval': '-',
    'Number of Correct Protocol': '-',
    'Protocol/Retrieval': '-'
}]
index = ['LLama 4 Maverick']

df_results = pd.DataFrame(data_results, index=index)

### LLAMA WITH RAG

In [ ]:
#Evaluation of Retrieval
# Function to normalize the protocol names 
def normalize_name(name):
    # Convert to lowercase
    name = name.lower()
    # Normalize spaces around dashes and parentheses, and remove extra spaces
    name = re.sub(r'\s*-\s*', '-', name)  # normalize spaces around dashes
    name = re.sub(r'\s*\(\s*', '(', name)  # normalize spaces before '('
    name = re.sub(r'\s*\)\s*', ')', name)  # normalize spaces after ')'
    name = re.sub(r"'", '', name)  # remove apostrophes
    name = re.sub(r'\s+', ' ', name).strip()  # Remove extra spaces
    return name

# Function to check if the normalized protocol name in ground truth is exactly contained in the normalized names of the corresponding row in results
def check_name_in_list1(list_one, list_two):
    results = []
    
    for i, name_one in enumerate(list_one):
        found = False
        normalized_name_one = normalize_name(name_one)
        
        if i < len(list_two):
            row_two = list_two[i]

            if isinstance(row_two, str):
                row_two = ast.literal_eval(row_two)  
            
            for name_two in row_two:
                normalized_name_two = normalize_name(name_two)
                if normalized_name_one == normalized_name_two:
                    found = True
                    break
        
        results.append(found)
    
    return results

def check_name_in_list2(list_one, list_two):
    results = []
    
    for i, name_one in enumerate(list_one):
        found = False
        normalized_name_one = normalize_name(name_one)
        
        if i < len(list_two):
            row_two = list_two[i]

            if isinstance(row_two, str):
                found = (normalized_name_one == normalize_name(row_two))
            else:
                # sonst wie bisher: über eine Liste iterieren
                for name_two in row_two:
                    normalized_name_two = normalize_name(name_two)
                    if normalized_name_one == normalized_name_two:
                        found = True
                        break
        
        results.append(found)
    
    return results

boolean_results = check_name_in_list1(df_GT['Protokolleinstufung'], df_OS_RAG['Retrieved Documents'])

boolean_results2 = check_name_in_list2(df_GT['Protokolleinstufung'], df_OS_RAG['Vorhersage Protokollname'])

correctly_retrieved = sum(boolean_results)

correct_protocol = sum(boolean_results2)

accuracy_correctprotocol_out_of_right_retrieval = correct_protocol/correctly_retrieved


In [ ]:
#MRI Sequences 
#Tokenize sequences
def tokenize(text):
    tokens = [token.strip() for token in text.split(',')]
    return tokens

#Function for calculation of token-based accuracy
def calculate_symmetric_token_based_accuracy(ground_truth, result):
    ground_truth_tokens = tokenize(ground_truth)
    result_tokens = tokenize(result)
    
    ground_truth_counts = Counter(ground_truth_tokens)
    result_counts = Counter(result_tokens)
    
    # Calculate the number of matching tokens considering repetitions
    matching_tokens_gt_to_res = sum(min(ground_truth_counts[token], result_counts[token]) for token in ground_truth_counts)
    matching_tokens_res_to_gt = sum(min(ground_truth_counts[token], result_counts[token]) for token in result_counts)
    
    # Total tokens in ground truth and result (considering repetitions)
    total_tokens_gt = sum(ground_truth_counts.values())
    total_tokens_res = sum(result_counts.values())
    
    # Calculate accuracy from ground truth to result
    accuracy_gt_to_res = matching_tokens_gt_to_res / total_tokens_gt if total_tokens_gt > 0 else 0
    
    # Calculate accuracy from result to ground truth
    accuracy_res_to_gt = matching_tokens_res_to_gt / total_tokens_res if total_tokens_res > 0 else 0
    
    # Symmetric accuracy: average of both directions
    symmetric_accuracy = (accuracy_gt_to_res + accuracy_res_to_gt) / 2
    
    return symmetric_accuracy

# Provide the data
ground_truths = df_GT['Sequenzen'] 
results2 = df_OS_RAG['Vorhersage Sequenzen']

accuracies_OS_RAG = []
for gt, res in zip(ground_truths, results2):
    accuracy = calculate_symmetric_token_based_accuracy(gt, res)
    accuracies_OS_RAG.append(accuracy)

#Calculate accuracy
average_accuracy_seq = sum(accuracies_OS_RAG) / len(accuracies_OS_RAG)

# Bootstrap function to calculate confidence intervals
def bootstrap_confidence_interval(data, num_samples=10000, confidence_level=0.95):
    # Resample with replacement and calculate means
    sample_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(num_samples)]

    # Calculate the percentiles for the given confidence level
    lower_bound = np.percentile(sample_means, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(sample_means, (1 + confidence_level) / 2 * 100)

    return lower_bound, upper_bound
confidence_interval_seq = bootstrap_confidence_interval(accuracies_OS_RAG)

#Contrast Medium Administration
#Replace values and change dtype
df_OS_RAG['Vorhersage Kontrastmittelgabe'] = df_OS_RAG['Vorhersage Kontrastmittelgabe'].replace({'ja': 1,'nein':0,'gegebenenfalls':0})
df_OS_RAG['Vorhersage Kontrastmittelgabe'] = df_OS_RAG['Vorhersage Kontrastmittelgabe'].astype(bool)

# Comparing the two columns and calculating accuracy
df_equal_OS_RAG = df_OS_RAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 

# Calculating the number of correct predictions (where the comparison is True)
correct_predictions = df_equal_OS_RAG.sum()

# Calculating accuracy
average_accuracy_cm = correct_predictions / len(df_equal_OS_RAG) * 100

# Set the number of bootstrap samples
n_iterations = 10000
bootstrap_accuracies = []

# Number of total samples
n = len(df_equal_OS_RAG)

# Perform bootstrapping to calculate confidence interval
for i in range(n_iterations):
    # Sample with replacement
    bootstrap_sample = np.random.choice(df_equal_OS_RAG, size=n, replace=True)
    
    # Calculate accuracy for this bootstrap sample
    accuracies = np.sum(bootstrap_sample) / len(bootstrap_sample) * 100
    bootstrap_accuracies.append(accuracies)

# Convert to a numpy array for easier manipulation
bootstrap_accuracies = np.array(bootstrap_accuracies)

# Calculate the 95% confidence interval
confidence_interval_cm = np.percentile(bootstrap_accuracies, [2.5, 97.5])

# Printing results

# New results to add
df_results.loc['Llama 4 Maverick RAG'] = {
    'Confidenz Interval Sequences': (confidence_interval_seq[0] * 100, confidence_interval_seq[1] * 100),
    'Average Accuracy Sequences': average_accuracy_seq * 100,
    'Kappa Score Sequences':'-',
    'Confidenz Interval Contrastmedium': (confidence_interval_cm[0], confidence_interval_cm[1]),
    'Average Accuracy Contrastmedium': average_accuracy_cm,
    'Kappa Score Contrastmedium':'-',
    'Number of Correct Retrieval': correctly_retrieved,
    'Number of Correct Protocol': correct_protocol,
    'Protocol/Retrieval':accuracy_correctprotocol_out_of_right_retrieval
}


### GPT

In [ ]:
#MRI sequences 
#Tokenize sequences
def tokenize(text):
    tokens = [token.strip() for token in text.split(',')]
    return tokens

#Function to calculate token-based accuracy
def calculate_symmetric_token_based_accuracy(ground_truth, result):
    ground_truth_tokens = tokenize(ground_truth)
    result_tokens = tokenize(result)
    
    ground_truth_counts = Counter(ground_truth_tokens)
    result_counts = Counter(result_tokens)
    
    # Calculate the number of matching tokens considering repetitions
    matching_tokens_gt_to_res = sum(min(ground_truth_counts[token], result_counts[token]) for token in ground_truth_counts)
    matching_tokens_res_to_gt = sum(min(ground_truth_counts[token], result_counts[token]) for token in result_counts)
    
    # Total tokens in ground truth and result (considering repetitions)
    total_tokens_gt = sum(ground_truth_counts.values())
    total_tokens_res = sum(result_counts.values())
    
    # Calculate accuracy from ground truth to result
    accuracy_gt_to_res = matching_tokens_gt_to_res / total_tokens_gt if total_tokens_gt > 0 else 0
    
    # Calculate accuracy from result to ground truth
    accuracy_res_to_gt = matching_tokens_res_to_gt / total_tokens_res if total_tokens_res > 0 else 0
    
    # Symmetric accuracy: average of both directions
    symmetric_accuracy = (accuracy_gt_to_res + accuracy_res_to_gt) / 2
    
    return symmetric_accuracy

# Provide the data
ground_truths = df_GT['Sequenzen'] 
results3 = df_woRAG['Vorhersage Sequenzen']

accuracies_woRAG = []
for gt, res in zip(ground_truths, results3):
    accuracy = calculate_symmetric_token_based_accuracy(gt, res)
    accuracies_woRAG.append(accuracy)

#Calculate accuracy
average_accuracy_seq = sum(accuracies_woRAG) / len(accuracies_woRAG)

# Bootstrap function to calculate confidence intervals
def bootstrap_confidence_interval(data, num_samples=10000, confidence_level=0.95):
    # Resample with replacement and calculate means
    sample_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(num_samples)]

    # Calculate the percentiles for the given confidence level
    lower_bound = np.percentile(sample_means, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(sample_means, (1 + confidence_level) / 2 * 100)

    return lower_bound, upper_bound
confidence_interval_seq = bootstrap_confidence_interval(accuracies_woRAG)

#Contrast Medium Administration
#Replace values and change dtype
df_woRAG['Vorhersage Kontrastmittelgabe'] = df_woRAG['Vorhersage Kontrastmittelgabe'].replace({'ja': 1,'nein':0,'gegebenenfalls':0})
df_woRAG['Vorhersage Kontrastmittelgabe'] = df_woRAG['Vorhersage Kontrastmittelgabe'].astype(bool)

# Comparing the two columns and calculating accuracy
df_equal_woRAG = df_woRAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 

# Calculating the number of correct predictions (where the comparison is True)
correct_predictions = df_equal_woRAG.sum()

# Calculating accuracy
average_accuracy_cm = correct_predictions / len(df_equal_woRAG) * 100

# Set the number of bootstrap samples
n_iterations = 10000
bootstrap_accuracies = []

# Number of total samples
n = len(df_equal_woRAG)

# Perform bootstrapping
for i in range(n_iterations):
    # Sample with replacement
    bootstrap_sample = np.random.choice(df_equal_woRAG, size=n, replace=True)
    
    # Calculate accuracy for this bootstrap sample
    accuracies = np.sum(bootstrap_sample) / len(bootstrap_sample) * 100
    bootstrap_accuracies.append(accuracies)

# Convert to a numpy array for easier manipulation
bootstrap_accuracies = np.array(bootstrap_accuracies)

# Calculate the 95% confidence interval
confidence_interval_cm = np.percentile(bootstrap_accuracies, [2.5, 97.5])

# New results to add
df_results.loc['GPT-5.2'] = {
    'Confidenz Interval Sequences': (confidence_interval_seq[0] * 100, confidence_interval_seq[1] * 100),
    'Average Accuracy Sequences': average_accuracy_seq * 100,
    'Kappa Score Sequences':'-',
    'Confidenz Interval Contrastmedium': (confidence_interval_cm[0], confidence_interval_cm[1]),
    'Average Accuracy Contrastmedium': average_accuracy_cm,
    'Kappa Score Contrastmedium':'-',
    'Number of Correct Retrieval': '-',
    'Number of Correct Protocol': '-',
    'Protocol/Retrieval': '-'
}

### GPT WITH RAG

In [ ]:
#Evaluation of Retrieval
# Function to normalize the protocol names 
def normalize_name(name):
    # Convert to lowercase
    name = name.lower()
    # Normalize spaces around dashes and parentheses, and remove extra spaces
    name = re.sub(r'\s*-\s*', '-', name)  # normalize spaces around dashes
    name = re.sub(r'\s*\(\s*', '(', name)  # normalize spaces before '('
    name = re.sub(r'\s*\)\s*', ')', name)  # normalize spaces after ')'
    name = re.sub(r"'", '', name)  # remove apostrophes
    name = re.sub(r'\s+', ' ', name).strip()  # Remove extra spaces
    return name

# Function to check if the normalized protocol name in ground truth is exactly contained in the normalized names of the corresponding row in results
def check_name_in_list1(list_one, list_two):
    results = []
    
    for i, name_one in enumerate(list_one):
        found = False
        normalized_name_one = normalize_name(name_one)
        
        if i < len(list_two):
            row_two = list_two[i]

            if isinstance(row_two, str):
                row_two = ast.literal_eval(row_two)  
            
            for name_two in row_two:
                normalized_name_two = normalize_name(name_two)
                if normalized_name_one == normalized_name_two:
                    found = True
                    break
        
        results.append(found)
    
    return results

def check_name_in_list2(list_one, list_two):
    results = []
    
    for i, name_one in enumerate(list_one):
        found = False
        normalized_name_one = normalize_name(name_one)
        
        if i < len(list_two):
            row_two = list_two[i]

            if isinstance(row_two, str):
                found = (normalized_name_one == normalize_name(row_two))
            else:
                # sonst wie bisher: über eine Liste iterieren
                for name_two in row_two:
                    normalized_name_two = normalize_name(name_two)
                    if normalized_name_one == normalized_name_two:
                        found = True
                        break
        
        results.append(found)
    
    return results

boolean_results = check_name_in_list1(df_GT['Protokolleinstufung'], df_RAG['Retrieved Documents'])

boolean_results2 = check_name_in_list2(df_GT['Protokolleinstufung'], df_RAG['Vorhersage Protokollname'])

correctly_retrieved = sum(boolean_results)

correct_protocol = sum(boolean_results2)

accuracy_correctprotocol_out_of_right_retrieval = correct_protocol/correctly_retrieved

In [ ]:
#MRI Sequences 
#Tokenize data
def tokenize(text):
    tokens = [token.strip() for token in text.split(',')]
    return tokens

#Function to calculate token-based accuracy
def calculate_symmetric_token_based_accuracy(ground_truth, result):
    ground_truth_tokens = tokenize(ground_truth)
    result_tokens = tokenize(result)
    
    ground_truth_counts = Counter(ground_truth_tokens)
    result_counts = Counter(result_tokens)
    
    # Calculate the number of matching tokens considering repetitions
    matching_tokens_gt_to_res = sum(min(ground_truth_counts[token], result_counts[token]) for token in ground_truth_counts)
    matching_tokens_res_to_gt = sum(min(ground_truth_counts[token], result_counts[token]) for token in result_counts)
    
    # Total tokens in ground truth and result (considering repetitions)
    total_tokens_gt = sum(ground_truth_counts.values())
    total_tokens_res = sum(result_counts.values())
    
    # Calculate accuracy from ground truth to result
    accuracy_gt_to_res = matching_tokens_gt_to_res / total_tokens_gt if total_tokens_gt > 0 else 0
    
    # Calculate accuracy from result to ground truth
    accuracy_res_to_gt = matching_tokens_res_to_gt / total_tokens_res if total_tokens_res > 0 else 0
    
    # Symmetric accuracy: average of both directions
    symmetric_accuracy = (accuracy_gt_to_res + accuracy_res_to_gt) / 2
    
    return symmetric_accuracy

# Provide the data
ground_truths = df_GT['Sequenzen'] 
results4 = df_RAG['Vorhersage Sequenzen']

accuracies_RAG = []
for gt, res in zip(ground_truths, results4):
    accuracy = calculate_symmetric_token_based_accuracy(gt, res)
    accuracies_RAG.append(accuracy)

#Calculate accuracy
average_accuracy_seq = sum(accuracies_RAG) / len(accuracies_RAG)

# Bootstrap function to calculate confidence intervals
def bootstrap_confidence_interval(data, num_samples=10000, confidence_level=0.95):
    # Resample with replacement and calculate means
    sample_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(num_samples)]

    # Calculate the percentiles for the given confidence level
    lower_bound = np.percentile(sample_means, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(sample_means, (1 + confidence_level) / 2 * 100)

    return lower_bound, upper_bound
confidence_interval_seq = bootstrap_confidence_interval(accuracies_RAG)

#Contrast Medium Administration
#Replace values and change dtype
df_RAG['Vorhersage Kontrastmittelgabe'] = df_RAG['Vorhersage Kontrastmittelgabe'].replace({'ja': 1,'nein':0,'gegebenenfalls':0})
df_RAG['Vorhersage Kontrastmittelgabe'] = df_RAG['Vorhersage Kontrastmittelgabe'].astype(bool)

# Comparing the two columns and calculating accuracy
df_equal_RAG = df_RAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 

# Calculating the number of correct predictions (where the comparison is True)
correct_predictions = df_equal_RAG.sum()

# Calculating accuracy
average_accuracy_cm = correct_predictions / len(df_equal_RAG) * 100

# Set the number of bootstrap samples
n_iterations = 10000
bootstrap_accuracies = []

# Number of total samples
n = len(df_equal_RAG)

# Perform bootstrapping
for i in range(n_iterations):
    # Sample with replacement
    bootstrap_sample = np.random.choice(df_equal_RAG, size=n, replace=True)
    
    # Calculate accuracy for this bootstrap sample
    accuracies = np.sum(bootstrap_sample) / len(bootstrap_sample) * 100
    bootstrap_accuracies.append(accuracies)

# Convert to a numpy array for easier manipulation
bootstrap_accuracies = np.array(bootstrap_accuracies)

# Calculate the 95% confidence interval
confidence_interval_cm = np.percentile(bootstrap_accuracies, [2.5, 97.5])

# New results to add
df_results.loc['GPT-5.2 RAG'] = {
    'Confidenz Interval Sequences': (confidence_interval_seq[0] * 100, confidence_interval_seq[1] * 100),
    'Average Accuracy Sequences': average_accuracy_seq * 100,
    'Kappa Score Sequences':'-',
    'Confidenz Interval Contrastmedium': (confidence_interval_cm[0], confidence_interval_cm[1]),
    'Average Accuracy Contrastmedium': average_accuracy_cm,
    'Kappa Score Contrastmedium':'-',
    'Number of Correct Retrieval': correctly_retrieved,
    'Number of Correct Protocol': correct_protocol,
    'Protocol/Retrieval':accuracy_correctprotocol_out_of_right_retrieval
}

### CLAUDE

In [ ]:
#MRI sequences 
#Tokenize sequences
def tokenize(text):
    tokens = [token.strip() for token in text.split(',')]
    return tokens

#Function to calculate token-based accuracy
def calculate_symmetric_token_based_accuracy(ground_truth, result):
    ground_truth_tokens = tokenize(ground_truth)
    result_tokens = tokenize(result)
    
    ground_truth_counts = Counter(ground_truth_tokens)
    result_counts = Counter(result_tokens)
    
    # Calculate the number of matching tokens considering repetitions
    matching_tokens_gt_to_res = sum(min(ground_truth_counts[token], result_counts[token]) for token in ground_truth_counts)
    matching_tokens_res_to_gt = sum(min(ground_truth_counts[token], result_counts[token]) for token in result_counts)
    
    # Total tokens in ground truth and result (considering repetitions)
    total_tokens_gt = sum(ground_truth_counts.values())
    total_tokens_res = sum(result_counts.values())
    
    # Calculate accuracy from ground truth to result
    accuracy_gt_to_res = matching_tokens_gt_to_res / total_tokens_gt if total_tokens_gt > 0 else 0
    
    # Calculate accuracy from result to ground truth
    accuracy_res_to_gt = matching_tokens_res_to_gt / total_tokens_res if total_tokens_res > 0 else 0
    
    # Symmetric accuracy: average of both directions
    symmetric_accuracy = (accuracy_gt_to_res + accuracy_res_to_gt) / 2
    
    return symmetric_accuracy

# Provide the data
ground_truths = df_GT['Sequenzen'] 
results3 = df_claude_woRAG['Vorhersage Sequenzen']

accuracies_claude_woRAG = []
for gt, res in zip(ground_truths, results3):
    accuracy = calculate_symmetric_token_based_accuracy(gt, res)
    accuracies_claude_woRAG.append(accuracy)

#Calculate accuracy
average_accuracy_seq = sum(accuracies_claude_woRAG) / len(accuracies_claude_woRAG)

# Bootstrap function to calculate confidence intervals
def bootstrap_confidence_interval(data, num_samples=10000, confidence_level=0.95):
    # Resample with replacement and calculate means
    sample_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(num_samples)]

    # Calculate the percentiles for the given confidence level
    lower_bound = np.percentile(sample_means, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(sample_means, (1 + confidence_level) / 2 * 100)

    return lower_bound, upper_bound
confidence_interval_seq = bootstrap_confidence_interval(accuracies_claude_woRAG)

#Contrast Medium Administration
#Replace values and change dtype
df_claude_woRAG['Vorhersage Kontrastmittelgabe'] = df_claude_woRAG['Vorhersage Kontrastmittelgabe'].replace({'ja': 1,'nein':0,'gegebenenfalls':0})
df_claude_woRAG['Vorhersage Kontrastmittelgabe'] = df_claude_woRAG['Vorhersage Kontrastmittelgabe'].astype(bool)

# Comparing the two columns and calculating accuracy
df_equal_woRAG = df_claude_woRAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 

# Calculating the number of correct predictions (where the comparison is True)
correct_predictions = df_equal_woRAG.sum()

# Calculating accuracy
average_accuracy_cm = correct_predictions / len(df_equal_woRAG) * 100

# Set the number of bootstrap samples
n_iterations = 10000
bootstrap_accuracies = []

# Number of total samples
n = len(df_equal_woRAG)

# Perform bootstrapping
for i in range(n_iterations):
    # Sample with replacement
    bootstrap_sample = np.random.choice(df_equal_woRAG, size=n, replace=True)
    
    # Calculate accuracy for this bootstrap sample
    accuracies = np.sum(bootstrap_sample) / len(bootstrap_sample) * 100
    bootstrap_accuracies.append(accuracies)

# Convert to a numpy array for easier manipulation
bootstrap_accuracies = np.array(bootstrap_accuracies)

# Calculate the 95% confidence interval
confidence_interval_cm = np.percentile(bootstrap_accuracies, [2.5, 97.5])

# New results to add
df_results.loc['Claude Opus 4.6'] = {
    'Confidenz Interval Sequences': (confidence_interval_seq[0] * 100, confidence_interval_seq[1] * 100),
    'Average Accuracy Sequences': average_accuracy_seq * 100,
    'Kappa Score Sequences':'-',
    'Confidenz Interval Contrastmedium': (confidence_interval_cm[0], confidence_interval_cm[1]),
    'Average Accuracy Contrastmedium': average_accuracy_cm,
    'Kappa Score Contrastmedium':'-',
    'Number of Correct Retrieval': '-',
    'Number of Correct Protocol': '-',
    'Protocol/Retrieval': '-'
}

### CLAUDE with RAG

In [ ]:
#Evaluation of Retrieval
# Function to normalize the protocol names 
def normalize_name(name):
    # Convert to lowercase
    name = name.lower()
    # Normalize spaces around dashes and parentheses, and remove extra spaces
    name = re.sub(r'\s*-\s*', '-', name)  # normalize spaces around dashes
    name = re.sub(r'\s*\(\s*', '(', name)  # normalize spaces before '('
    name = re.sub(r'\s*\)\s*', ')', name)  # normalize spaces after ')'
    name = re.sub(r"'", '', name)  # remove apostrophes
    name = re.sub(r'\s+', ' ', name).strip()  # Remove extra spaces
    return name

# Function to check if the normalized protocol name in ground truth is exactly contained in the normalized names of the corresponding row in results
def check_name_in_list1(list_one, list_two):
    results = []
    
    for i, name_one in enumerate(list_one):
        found = False
        normalized_name_one = normalize_name(name_one)
        
        if i < len(list_two):
            row_two = list_two[i]

            if isinstance(row_two, str):
                row_two = ast.literal_eval(row_two)  
            
            for name_two in row_two:
                normalized_name_two = normalize_name(name_two)
                if normalized_name_one == normalized_name_two:
                    found = True
                    break
        
        results.append(found)
    
    return results

def check_name_in_list2(list_one, list_two):
    results = []
    
    for i, name_one in enumerate(list_one):
        found = False
        normalized_name_one = normalize_name(name_one)
        
        if i < len(list_two):
            row_two = list_two[i]

            if isinstance(row_two, str):
                found = (normalized_name_one == normalize_name(row_two))
            else:
                # sonst wie bisher: über eine Liste iterieren
                for name_two in row_two:
                    normalized_name_two = normalize_name(name_two)
                    if normalized_name_one == normalized_name_two:
                        found = True
                        break
        
        results.append(found)
    
    return results

boolean_results = check_name_in_list1(df_GT['Protokolleinstufung'], df_claude_RAG['Retrieved Documents'])

boolean_results2 = check_name_in_list2(df_GT['Protokolleinstufung'], df_claude_RAG['Vorhersage Protokollname'])

correctly_retrieved = sum(boolean_results)

correct_protocol = sum(boolean_results2)

accuracy_correctprotocol_out_of_right_retrieval = correct_protocol/correctly_retrieved

In [ ]:
#MRI Sequences 
#Tokenize data
def tokenize(text):
    tokens = [token.strip() for token in text.split(',')]
    return tokens

#Function to calculate token-based accuracy
def calculate_symmetric_token_based_accuracy(ground_truth, result):
    ground_truth_tokens = tokenize(ground_truth)
    result_tokens = tokenize(result)
    
    ground_truth_counts = Counter(ground_truth_tokens)
    result_counts = Counter(result_tokens)
    
    # Calculate the number of matching tokens considering repetitions
    matching_tokens_gt_to_res = sum(min(ground_truth_counts[token], result_counts[token]) for token in ground_truth_counts)
    matching_tokens_res_to_gt = sum(min(ground_truth_counts[token], result_counts[token]) for token in result_counts)
    
    # Total tokens in ground truth and result (considering repetitions)
    total_tokens_gt = sum(ground_truth_counts.values())
    total_tokens_res = sum(result_counts.values())
    
    # Calculate accuracy from ground truth to result
    accuracy_gt_to_res = matching_tokens_gt_to_res / total_tokens_gt if total_tokens_gt > 0 else 0
    
    # Calculate accuracy from result to ground truth
    accuracy_res_to_gt = matching_tokens_res_to_gt / total_tokens_res if total_tokens_res > 0 else 0
    
    # Symmetric accuracy: average of both directions
    symmetric_accuracy = (accuracy_gt_to_res + accuracy_res_to_gt) / 2
    
    return symmetric_accuracy

# Provide the data
ground_truths = df_GT['Sequenzen'] 
results4 = df_claude_RAG['Vorhersage Sequenzen']

accuracies_claude_RAG = []
for gt, res in zip(ground_truths, results4):
    accuracy = calculate_symmetric_token_based_accuracy(gt, res)
    accuracies_claude_RAG.append(accuracy)

#Calculate accuracy
average_accuracy_seq = sum(accuracies_claude_RAG) / len(accuracies_claude_RAG)

# Bootstrap function to calculate confidence intervals
def bootstrap_confidence_interval(data, num_samples=10000, confidence_level=0.95):
    # Resample with replacement and calculate means
    sample_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(num_samples)]

    # Calculate the percentiles for the given confidence level
    lower_bound = np.percentile(sample_means, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(sample_means, (1 + confidence_level) / 2 * 100)

    return lower_bound, upper_bound
confidence_interval_seq = bootstrap_confidence_interval(accuracies_claude_RAG)

#Contrast Medium Administration
#Replace values and change dtype
df_claude_RAG['Vorhersage Kontrastmittelgabe'] = df_claude_RAG['Vorhersage Kontrastmittelgabe'].replace({'ja': 1,'nein':0,'gegebenenfalls':0})
df_claude_RAG['Vorhersage Kontrastmittelgabe'] = df_claude_RAG['Vorhersage Kontrastmittelgabe'].astype(bool)

# Comparing the two columns and calculating accuracy
df_equal_RAG = df_claude_RAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 

# Calculating the number of correct predictions (where the comparison is True)
correct_predictions = df_equal_RAG.sum()

# Calculating accuracy
average_accuracy_cm = correct_predictions / len(df_equal_RAG) * 100

# Set the number of bootstrap samples
n_iterations = 10000
bootstrap_accuracies = []

# Number of total samples
n = len(df_equal_RAG)

# Perform bootstrapping
for i in range(n_iterations):
    # Sample with replacement
    bootstrap_sample = np.random.choice(df_equal_RAG, size=n, replace=True)
    
    # Calculate accuracy for this bootstrap sample
    accuracies = np.sum(bootstrap_sample) / len(bootstrap_sample) * 100
    bootstrap_accuracies.append(accuracies)

# Convert to a numpy array for easier manipulation
bootstrap_accuracies = np.array(bootstrap_accuracies)

# Calculate the 95% confidence interval
confidence_interval_cm = np.percentile(bootstrap_accuracies, [2.5, 97.5])

# New results to add
df_results.loc['Claude Opus 4.6 RAG'] = {
    'Confidenz Interval Sequences': (confidence_interval_seq[0] * 100, confidence_interval_seq[1] * 100),
    'Average Accuracy Sequences': average_accuracy_seq * 100,
    'Kappa Score Sequences':'-',
    'Confidenz Interval Contrastmedium': (confidence_interval_cm[0], confidence_interval_cm[1]),
    'Average Accuracy Contrastmedium': average_accuracy_cm,
    'Kappa Score Contrastmedium':'-',
    'Number of Correct Retrieval': correctly_retrieved,
    'Number of Correct Protocol': correct_protocol,
    'Protocol/Retrieval':accuracy_correctprotocol_out_of_right_retrieval
}

In [ ]:
df_results.to_csv(path_or_buf='/path/results.csv',sep=';')

## SEQUENCE EVALUATION

In [ ]:
#Load results
df_1 = pd.read_csv(filepath_or_buffer='/path/results.csv',sep=';')
df_1_RAG = pd.read_csv(filepath_or_buffer='/path/results.csv',sep=';')

#Import Ground Truth
df_GT = pd.read_csv(filepath_or_buffer='/Users/laranoellereiner/Documents/Forschungsprojekte/Automated MRI Protocoling/Data/MRI data/MRI data classification/MRT_Einstufungen_LLM_Juliane.csv', sep=';')

In [ ]:
#Tokenize sequences
def tokenize(text):
    tokens = [token.strip() for token in text.split(',')]
    return tokens

#Function for calculation of token-based accuracy
def calculate_symmetric_token_based_accuracy(ground_truth, result):
    ground_truth_tokens = tokenize(ground_truth)
    result_tokens = tokenize(result)
    
    ground_truth_counts = Counter(ground_truth_tokens)
    result_counts = Counter(result_tokens)
    
    # Calculate the number of matching tokens considering repetitions
    matching_tokens_gt_to_res = sum(min(ground_truth_counts[token], result_counts[token]) for token in ground_truth_counts)
    matching_tokens_res_to_gt = sum(min(ground_truth_counts[token], result_counts[token]) for token in result_counts)
    
    # Total tokens in ground truth and result (considering repetitions)
    total_tokens_gt = sum(ground_truth_counts.values())
    total_tokens_res = sum(result_counts.values())
    
    # Calculate accuracy from ground truth to result
    accuracy_gt_to_res = matching_tokens_gt_to_res / total_tokens_gt if total_tokens_gt > 0 else 0
    
    # Calculate accuracy from result to ground truth
    accuracy_res_to_gt = matching_tokens_res_to_gt / total_tokens_res if total_tokens_res > 0 else 0
    
    # Symmetric accuracy: average of both directions
    symmetric_accuracy = (accuracy_gt_to_res + accuracy_res_to_gt) / 2
    
    return symmetric_accuracy

# Provide the data
ground_truths = df_GT['Sequenzen'] 
results1 = df_1['Vorhersage Sequenzen']

accuracies_woRAG = []
for gt, res in zip(ground_truths, results1):
    accuracy = calculate_symmetric_token_based_accuracy(gt, res)
    accuracies_woRAG.append(accuracy)

#Calculate average accuracy
average_accuracy_seq = sum(accuracies_woRAG) / len(accuracies_woRAG)

#Calculate missing and redundant sequences
missing_counts_per_row = []
redundant_counts_per_row = []
missing_n_per_row = []
redundant_n_per_row = []

missing_global = Counter()
redundant_global = Counter()

for gt, res in zip(ground_truths, results1):
    gt_tokens = tokenize(gt)
    res_tokens = tokenize(res)

    gt_counts = Counter(gt_tokens)
    res_counts = Counter(res_tokens)

    # Missing: GT - RES 
    missing_counts = Counter()
    for tok, c in gt_counts.items():
        diff = c - res_counts.get(tok, 0)
        if diff > 0:
            missing_counts[tok] = diff

    # Redundant: RES - GT 
    redundant_counts = Counter()
    for tok, c in res_counts.items():
        diff = c - gt_counts.get(tok, 0)
        if diff > 0:
            redundant_counts[tok] = diff

    missing_counts_per_row.append(missing_counts)
    redundant_counts_per_row.append(redundant_counts)

    missing_n = sum(missing_counts.values())
    redundant_n = sum(redundant_counts.values())

    missing_n_per_row.append(missing_n)
    redundant_n_per_row.append(redundant_n)

    missing_global.update(missing_counts)
    redundant_global.update(redundant_counts)

Total_Missing_Sequences = int(sum(missing_n_per_row))
Total_Redundant_Sequences = int(sum(redundant_n_per_row))

Mean_Missing_Sequences = float(np.mean(missing_n_per_row)) if len(missing_n_per_row) > 0 else 0.0
Mean_Redundant_Sequences = float(np.mean(redundant_n_per_row)) if len(redundant_n_per_row) > 0 else 0.0

#Calculate most common missing and redundant sequences
Missing_Sequences_Details = dict(missing_global.most_common(20))
Redundant_Sequences_Details = dict(redundant_global.most_common(20))


data_results = [{
    'Confidenz Interval Sequences': (confidence_interval_seq[0] * 100, confidence_interval_seq[1] * 100),
    'Average Accuracy Sequences': average_accuracy_seq * 100,
    'Total Missing Sequences': Total_Missing_Sequences,
    'Total Redundant Sequences': Total_Redundant_Sequences,
    'Mean Missing Sequences': Mean_Missing_Sequences,
    'Mean Redundant Sequences': Mean_Redundant_Sequences,
    'Missing Sequences Details': Missing_Sequences_Details,
    'Redundant Sequences Details': Redundant_Sequences_Details,
}]
index = ['LLama 4 Maverick']

df_results = pd.DataFrame(data_results, index=index)

In [ ]:
#Tokenize sequences
def tokenize(text):
    tokens = [token.strip() for token in text.split(',')]
    return tokens

#Function for calculation of token-based accuracy
def calculate_symmetric_token_based_accuracy(ground_truth, result):
    ground_truth_tokens = tokenize(ground_truth)
    result_tokens = tokenize(result)
    
    ground_truth_counts = Counter(ground_truth_tokens)
    result_counts = Counter(result_tokens)
    
    # Calculate the number of matching tokens considering repetitions
    matching_tokens_gt_to_res = sum(min(ground_truth_counts[token], result_counts[token]) for token in ground_truth_counts)
    matching_tokens_res_to_gt = sum(min(ground_truth_counts[token], result_counts[token]) for token in result_counts)
    
    # Total tokens in ground truth and result (considering repetitions)
    total_tokens_gt = sum(ground_truth_counts.values())
    total_tokens_res = sum(result_counts.values())
    
    # Calculate accuracy from ground truth to result
    accuracy_gt_to_res = matching_tokens_gt_to_res / total_tokens_gt if total_tokens_gt > 0 else 0
    
    # Calculate accuracy from result to ground truth
    accuracy_res_to_gt = matching_tokens_res_to_gt / total_tokens_res if total_tokens_res > 0 else 0
    
    # Symmetric accuracy: average of both directions
    symmetric_accuracy = (accuracy_gt_to_res + accuracy_res_to_gt) / 2
    
    return symmetric_accuracy

# Provide the data
ground_truths = df_GT['Sequenzen'] 
results2 = df_1_RAG['Vorhersage Sequenzen']

accuracies_RAG = []
for gt, res in zip(ground_truths, results1):
    accuracy = calculate_symmetric_token_based_accuracy(gt, res)
    accuracies_RAG.append(accuracy)

#Calculate average accuracy
average_accuracy_seq = sum(accuracies_RAG) / len(accuracies_RAG)

#Calculate missing and redundant sequences
missing_counts_per_row = []
redundant_counts_per_row = []
missing_n_per_row = []
redundant_n_per_row = []

missing_global = Counter()
redundant_global = Counter()

for gt, res in zip(ground_truths, results1):
    gt_tokens = tokenize(gt)
    res_tokens = tokenize(res)

    gt_counts = Counter(gt_tokens)
    res_counts = Counter(res_tokens)

    # Missing: GT - RES 
    missing_counts = Counter()
    for tok, c in gt_counts.items():
        diff = c - res_counts.get(tok, 0)
        if diff > 0:
            missing_counts[tok] = diff

    # Redundant: RES - GT 
    redundant_counts = Counter()
    for tok, c in res_counts.items():
        diff = c - gt_counts.get(tok, 0)
        if diff > 0:
            redundant_counts[tok] = diff

    missing_counts_per_row.append(missing_counts)
    redundant_counts_per_row.append(redundant_counts)

    missing_n = sum(missing_counts.values())
    redundant_n = sum(redundant_counts.values())

    missing_n_per_row.append(missing_n)
    redundant_n_per_row.append(redundant_n)

    missing_global.update(missing_counts)
    redundant_global.update(redundant_counts)

Total_Missing_Sequences = int(sum(missing_n_per_row))
Total_Redundant_Sequences = int(sum(redundant_n_per_row))

Mean_Missing_Sequences = float(np.mean(missing_n_per_row)) if len(missing_n_per_row) > 0 else 0.0
Mean_Redundant_Sequences = float(np.mean(redundant_n_per_row)) if len(redundant_n_per_row) > 0 else 0.0

#Calculate most common missing and redundant sequences
Missing_Sequences_Details = dict(missing_global.most_common(20))
Redundant_Sequences_Details = dict(redundant_global.most_common(20))

df_results.loc['Llama 4 Maverick RAG'] = {
    'Confidenz Interval Sequences': (confidence_interval_seq[0] * 100, confidence_interval_seq[1] * 100),
    'Average Accuracy Sequences': average_accuracy_seq * 100,
    'Total Missing Sequences': Total_Missing_Sequences,
    'Total Redundant Sequences': Total_Redundant_Sequences,
    'Mean Missing Sequences': Mean_Missing_Sequences,
    'Mean Redundant Sequences': Mean_Redundant_Sequences,
    'Missing Sequences Details': Missing_Sequences_Details,
    'Redundant Sequences Details': Redundant_Sequences_Details,
}

In [ ]:
df_results.to_csv(path_or_buf='/path/results.csv',sep=';')

## EMBEDDING MODELS

In [ ]:
#Load results
df_ada_002 = pd.read_csv(filepath_or_buffer='path/prediction ada 002.csv',sep=';')
df_3large =  pd.read_csv(filepath_or_buffer='path/prediction 3-large.csv',sep=';')

#Import Ground Truth
df_GT = pd.read_csv(filepath_or_buffer='path/ground truth.csv', sep=';')

In [ ]:
#Evaluation of Retrieval
#Function to normalize the protocol names 
def normalize_name(name):
    name = name.lower()
    name = re.sub(r'\s*-\s*', '-', name)  
    name = re.sub(r'\s*\(\s*', '(', name)  
    name = re.sub(r'\s*\)\s*', ')', name) 
    name = re.sub(r"'", '', name)  
    name = re.sub(r'\s+', ' ', name).strip()
    return name

#Function: if ground truth protocol name appears in results
def check_name_in_list1(list_one, list_two):
    results = []
    
    for i, name_one in enumerate(list_one):
        found = False
        normalized_name_one = normalize_name(name_one)
        
        if i < len(list_two):
            row_two = list_two[i]

            if isinstance(row_two, str):
                row_two = ast.literal_eval(row_two)  
            
            for name_two in row_two:
                normalized_name_two = normalize_name(name_two)
                if normalized_name_one == normalized_name_two:
                    found = True
                    break
        
        results.append(found)
    
    return results


boolean_results_ada = check_name_in_list1(df_GT['Protokolleinstufung'], df_ada_002['Retrieved Documents'])

correctly_retrieved_ada = sum(boolean_results_ada)

boolean_results_3l = check_name_in_list1(df_GT['Protokolleinstufung'], df_3large['Retrieved Documents'])

correctly_retrieved_3l = sum(boolean_results_3l)

#Bootstrap-CI 
def bootstrap_ci_rate(boolean_list, n_bootstrap=10000, ci=95, seed=42):
    rng = np.random.default_rng(seed)
    data = np.asarray(boolean_list, dtype=int)
    n = len(data)

    idx = rng.integers(0, n, size=(n_bootstrap, n))
    boot_means = data[idx].mean(axis=1)

    alpha = (100 - ci) / 2
    lower = np.percentile(boot_means, alpha)
    upper = np.percentile(boot_means, 100 - alpha)

    return data.mean(), lower, upper


#Paired Bootstrap 
def paired_bootstrap_ci_diff(boolean_a, boolean_b, n_bootstrap=10000, ci=95, seed=42):
    # diff = mean(b - a) pro bootstrap sample
    rng = np.random.default_rng(seed)
    a = np.asarray(boolean_a, dtype=int)
    b = np.asarray(boolean_b, dtype=int)

    if len(a) != len(b):
        raise ValueError("Beide Boolean-Listen müssen gleich lang sein (gleiche Queries).")

    n = len(a)
    idx = rng.integers(0, n, size=(n_bootstrap, n))

    boot_diff = (b[idx].mean(axis=1) - a[idx].mean(axis=1))
    alpha = (100 - ci) / 2
    lower = np.percentile(boot_diff, alpha)
    upper = np.percentile(boot_diff, 100 - alpha)

    point = b.mean() - a.mean()
    return point, lower, upper


#McNemar-Test
def mcnemar_test(boolean_a, boolean_b, exact=False, correction=True):
    """
    a,b: bool lists
    exact=False: asymptotische Chi^2 Approximation (gut bei größeren discordant counts)
    exact=True : exact binomial test (gut bei kleinen discordant counts)
    """
    a = np.asarray(boolean_a, dtype=bool)
    b = np.asarray(boolean_b, dtype=bool)

    if len(a) != len(b):
        raise ValueError("Beide Boolean-Listen müssen gleich lang sein (gleiche Queries).")

    # Kontingenztafel (discordant pairs):
    # b01: a=False, b=True  (b gewinnt)
    # b10: a=True,  b=False (a gewinnt)
    b01 = np.sum((~a) & ( b))
    b10 = np.sum(( a) & (~b))

    if exact:
        # Exact binomial: unter H0 sind b01 und b10 gleich wahrscheinlich
        from math import comb
        n = b01 + b10
        if n == 0:
            return {"b01": int(b01), "b10": int(b10), "p_value": 1.0, "note": "Keine discordant pairs"}
        # zweiseitiger exakter Binomialtest: p=0.5
        # p_value = 2 * min(P(X<=min), P(X>=max))
        # X ~ Bin(n, 0.5)
        k = min(b01, b10)
        # P(X<=k)
        p_le = sum(comb(n, i) for i in range(0, k+1)) * (0.5 ** n)
        # P(X>=n-k)
        p_ge = sum(comb(n, i) for i in range(n-k, n+1)) * (0.5 ** n)
        p_value = 2 * min(p_le, p_ge)
        p_value = min(1.0, p_value)
        return {"b01": int(b01), "b10": int(b10), "p_value": float(p_value), "note": "Exact binomial McNemar"}

    else:
        # Chi^2 approx with optional continuity correction
        if b01 + b10 == 0:
            return {"b01": int(b01), "b10": int(b10), "p_value": 1.0, "note": "Keine discordant pairs"}

        import scipy.stats as st
        num = (abs(b01 - b10) - 1)**2 if correction else (b01 - b10)**2
        chi2 = num / (b01 + b10)
        p_value = 1 - st.chi2.cdf(chi2, df=1)
        return {"b01": int(b01), "b10": int(b10), "p_value": float(p_value), "note": "Chi^2 McNemar"}


mean_ada, lo_ada, hi_ada = bootstrap_ci_rate(boolean_results_ada, n_bootstrap=20000, ci=95, seed=1)
mean_3l,  lo_3l,  hi_3l  = bootstrap_ci_rate(boolean_results_3l,  n_bootstrap=20000, ci=95, seed=1)

diff_point, diff_lo, diff_hi = paired_bootstrap_ci_diff(
    boolean_results_ada, boolean_results_3l, n_bootstrap=20000, ci=95, seed=1
)

mcn = mcnemar_test(boolean_results_ada, boolean_results_3l, exact=False, correction=True)

n_queries = len(boolean_results_ada)

data_results = [{

    # k-Values
    'N Queries': n_queries,

    # Text-embedding-ada-002
    'ada-002 Hits (k)': correctly_retrieved_ada,
    'ada-002 Hit Rate': round(mean_ada, 3),
    'ada-002 Lower (95%)': round(lo_ada, 3),
    'ada-002 Upper (95%)': round(hi_ada, 3),

    # Text-embedding-3-large
    '3-Large Hits (k)': correctly_retrieved_3l,
    '3-Large Hit Rate': round(mean_3l, 3),
    '3-Large CI Lower (95%)': round(lo_3l, 3),
    '3-Large CI Upper (95%)': round(hi_3l, 3),

    # Modellvergleich (gepaart)
    'Difference (3L - ADA)': round(diff_point, 3),
    'Diff CI Lower (95%)': round(diff_lo, 3),
    'Diff CI Upper (95%)': round(diff_hi, 3),

    # McNemar
    'McNemar b01 (ADA F / 3L T)': mcn['b01'],
    'McNemar b10 (ADA T / 3L F)': mcn['b10'],
    'McNemar p-value': round(mcn['p_value'], 5)
}]

df_results = pd.DataFrame(data_results, index=['2'])

In [ ]:
df_results.to_csv(path_or_buf='/path/results.csv',sep=';')

## RADIOLOGISTS

In [ ]:
df1 = pd.read_csv(filepath_or_buffer='/path/GT',sep=';')
df2= pd.read_csv(filepath_or_buffer='/path/selection resident 2', sep=';')
df3 = pd.read_csv(filepath_or_buffer='/path/selection resident 1', sep=';')
df4 = pd.read_csv(filepath_or_buffer='/path/selection radiologist 2', sep=';')
df5  = pd.read_csv(filepath_or_buffer='path/selection radiologist 1', sep=';')
df_GT = df1.fillna('')
df_resident2 = df2.fillna('')
df_resident1 = df3.fillna('')
df_radiologist2 = df4.fillna('')
df_radiologist1 = df5.fillna('')

### Radiologist 1

In [ ]:
#MRI Sequences
#Tokenize sequences
def tokenize(text):
    tokens = [token.strip() for token in text.split(',')]
    return tokens

#Function to calculate token-based accuracy
def calculate_symmetric_token_based_accuracy(ground_truth, result):
    ground_truth_tokens = tokenize(ground_truth)
    result_tokens = tokenize(result)
    
    ground_truth_counts = Counter(ground_truth_tokens)
    result_counts = Counter(result_tokens)
    
    # Calculate the number of matching tokens considering repetitions
    matching_tokens_gt_to_res = sum(min(ground_truth_counts[token], result_counts[token]) for token in ground_truth_counts)
    matching_tokens_res_to_gt = sum(min(ground_truth_counts[token], result_counts[token]) for token in result_counts)
    
    # Total tokens in ground truth and result (considering repetitions)
    total_tokens_gt = sum(ground_truth_counts.values())
    total_tokens_res = sum(result_counts.values())
    
    # Calculate accuracy from ground truth to result
    accuracy_gt_to_res = matching_tokens_gt_to_res / total_tokens_gt if total_tokens_gt > 0 else 0
    
    # Calculate accuracy from result to ground truth
    accuracy_res_to_gt = matching_tokens_res_to_gt / total_tokens_res if total_tokens_res > 0 else 0
    
    # Symmetric accuracy: average of both directions
    symmetric_accuracy = (accuracy_gt_to_res + accuracy_res_to_gt) / 2
    
    return symmetric_accuracy

# Example usage with your provided data:
ground_truths = df_GT['Sequenzen']
results = df_radiologist1['Sequenzen'] 

accuracies_rad1 = []
for gt, res in zip(ground_truths, results):
    accuracy = calculate_symmetric_token_based_accuracy(gt, res)
    accuracies_rad1.append(accuracy)

average_accuracy_seq = sum(accuracies_rad1) / len(accuracies_rad1)

# Cohen's Kappa calculation
# Create agreement labels: 1 if they agree (accuracy == 1), else 0
agreement_labels_ground_truth = []
agreement_labels_result = []

for gt, res in zip(ground_truths, results):
    # Tokenize and sort to compare token-based categories
    tokenized_gt = tokenize(gt)
    tokenized_res = tokenize(res)
    
    # Label agreement: 1 if identical tokens, else 0
    agreement_labels_ground_truth.append(' '.join(sorted(tokenized_gt)))
    agreement_labels_result.append(' '.join(sorted(tokenized_res)))

# Calculate Cohen's Kappa
kappa_score_seq = cohen_kappa_score(agreement_labels_ground_truth, agreement_labels_result)

# Bootstrap function to calculate confidence intervals
def bootstrap_confidence_interval(data, num_samples=10000, confidence_level=0.95):
    # Resample with replacement and calculate means
    sample_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(num_samples)]

    # Calculate the percentiles for the given confidence level
    lower_bound = np.percentile(sample_means, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(sample_means, (1 + confidence_level) / 2 * 100)

    return lower_bound, upper_bound
confidence_interval_seq = bootstrap_confidence_interval(accuracies_rad1)

In [ ]:
#Contrast Medium Administration
# Comparing the two columns and calculating accuracy
df_radiologist1['Kontrastmittelgabe'] = df_radiologist1['Kontrastmittelgabe'].astype(bool)
df_GT['Kontrastmittelgabe'] = df_GT['Kontrastmittelgabe'].astype(bool)
df_equal_rad1 = df_radiologist1['Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe']

# Calculating the number of correct predictions (where the comparison is True)
correct_predictions = df_equal_rad1.sum()

# Calculating accuracy
average_accuracy_cm = correct_predictions / len(df_equal_rad1) * 100

# Set the number of bootstrap samples
n_iterations = 10000
bootstrap_accuracies = []

# Number of total samples
n = len(df_equal_rad1)

# Perform bootstrapping
for i in range(n_iterations):
    # Sample with replacement
    bootstrap_sample = np.random.choice(df_equal_rad1, size=n, replace=True)
    
    # Calculate accuracy for this bootstrap sample
    accuracies = np.sum(bootstrap_sample) / len(bootstrap_sample) * 100
    bootstrap_accuracies.append(accuracies)

# Convert to a numpy array for easier manipulation
bootstrap_accuracies = np.array(bootstrap_accuracies)

# Calculate the 95% confidence interval
confidence_interval_cm = np.percentile(bootstrap_accuracies, [2.5, 97.5])

# Calculate Cohen's Kappa
kappa_score_cm = cohen_kappa_score(df_radiologist1['Kontrastmittelgabe'], df_GT['Kontrastmittelgabe'])


# New results to add
df_results.loc['Radiologist 1'] = {
    'Confidenz Interval Sequences': (confidence_interval_seq[0] * 100, confidence_interval_seq[1] * 100),
    'Average Accuracy Sequences': average_accuracy_seq * 100,
    'Kappa Score Sequences':kappa_score_seq,
    'Confidenz Interval Contrastmedium': (confidence_interval_cm[0], confidence_interval_cm[1]),
    'Average Accuracy Contrastmedium': average_accuracy_cm,
    'Kappa Score Contrastmedium': kappa_score_cm,
    'Number of Correct Retrieval': '-',
    'Number of Correct Protocol': '-',
    'Protocol/Retrieval':'-'
}



### Radiologist 2

In [ ]:
#MRI Sequences 
#Tokenize sequences
def tokenize(text):
    tokens = [token.strip() for token in text.split(',')]
    return tokens

#Function to calculate token-based accuracy
def calculate_symmetric_token_based_accuracy(ground_truth, result):
    ground_truth_tokens = tokenize(ground_truth)
    result_tokens = tokenize(result)
    
    ground_truth_counts = Counter(ground_truth_tokens)
    result_counts = Counter(result_tokens)
    
    # Calculate the number of matching tokens considering repetitions
    matching_tokens_gt_to_res = sum(min(ground_truth_counts[token], result_counts[token]) for token in ground_truth_counts)
    matching_tokens_res_to_gt = sum(min(ground_truth_counts[token], result_counts[token]) for token in result_counts)
    
    # Total tokens in ground truth and result (considering repetitions)
    total_tokens_gt = sum(ground_truth_counts.values())
    total_tokens_res = sum(result_counts.values())
    
    # Calculate accuracy from ground truth to result
    accuracy_gt_to_res = matching_tokens_gt_to_res / total_tokens_gt if total_tokens_gt > 0 else 0
    
    # Calculate accuracy from result to ground truth
    accuracy_res_to_gt = matching_tokens_res_to_gt / total_tokens_res if total_tokens_res > 0 else 0
    
    # Symmetric accuracy: average of both directions
    symmetric_accuracy = (accuracy_gt_to_res + accuracy_res_to_gt) / 2
    
    return symmetric_accuracy

# Example usage with your provided data:
ground_truths = df_GT['Sequenzen']
results = df_radiologist2['Sequenzen'] 

accuracies_rad2 = []
for gt, res in zip(ground_truths, results):
    accuracy = calculate_symmetric_token_based_accuracy(gt, res)
    accuracies_rad2.append(accuracy)

average_accuracy_seq = sum(accuracies_rad2) / len(accuracies_rad2)

# Cohen's Kappa calculation
# Create agreement labels: 1 if they agree (accuracy == 1), else 0
agreement_labels_ground_truth = []
agreement_labels_result = []

for gt, res in zip(ground_truths, results):
    # Tokenize and sort to compare token-based categories
    tokenized_gt = tokenize(gt)
    tokenized_res = tokenize(res)
    
    # Label agreement: 1 if identical tokens, else 0
    agreement_labels_ground_truth.append(' '.join(sorted(tokenized_gt)))
    agreement_labels_result.append(' '.join(sorted(tokenized_res)))

# Calculate Cohen's Kappa
kappa_score_seq = cohen_kappa_score(agreement_labels_ground_truth, agreement_labels_result)

# Bootstrap function to calculate confidence intervals
def bootstrap_confidence_interval(data, num_samples=10000, confidence_level=0.95):
    # Resample with replacement and calculate means
    sample_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(num_samples)]

    # Calculate the percentiles for the given confidence level
    lower_bound = np.percentile(sample_means, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(sample_means, (1 + confidence_level) / 2 * 100)

    return lower_bound, upper_bound
confidence_interval_seq = bootstrap_confidence_interval(accuracies_rad2)

In [ ]:
#Contrast Medium Administration
# Comparing the two columns and calculating accuracy
df_equal_rad2 = df_radiologist2['Kontrastmittelgabe'].astype(bool) == df_GT['Kontrastmittelgabe'].astype(bool)

# Calculating the number of correct predictions (where the comparison is True)
correct_predictions = df_equal_rad2.sum()

# Calculating accuracy
average_accuracy_cm = correct_predictions / len(df_equal_rad2) * 100

# Set the number of bootstrap samples
n_iterations = 10000
bootstrap_accuracies = []

# Number of total samples
n = len(df_equal_rad2)

# Perform bootstrapping
for i in range(n_iterations):
    # Sample with replacement
    bootstrap_sample = np.random.choice(df_equal_rad2, size=n, replace=True)
    
    # Calculate accuracy for this bootstrap sample
    accuracies = np.sum(bootstrap_sample) / len(bootstrap_sample) * 100
    bootstrap_accuracies.append(accuracies)

# Convert to a numpy array for easier manipulation
bootstrap_accuracies = np.array(bootstrap_accuracies)

# Calculate the 95% confidence interval
confidence_interval_cm = np.percentile(bootstrap_accuracies, [2.5, 97.5])

# Calculate Cohen's Kappa
kappa_score_cm = cohen_kappa_score(df_radiologist2['Kontrastmittelgabe'].astype(bool), df_GT['Kontrastmittelgabe'].astype(bool))

# New results to add
df_results.loc['Radiologist 2'] = {
    'Confidenz Interval Sequences': (confidence_interval_seq[0] * 100, confidence_interval_seq[1] * 100),
    'Average Accuracy Sequences': average_accuracy_seq * 100,
    'Kappa Score Sequences':kappa_score_seq,
    'Confidenz Interval Contrastmedium': (confidence_interval_cm[0], confidence_interval_cm[1]),
    'Average Accuracy Contrastmedium': average_accuracy_cm,
    'Kappa Score Contrastmedium': kappa_score_cm,
    'Number of Correct Retrieval': '-',
    'Number of Correct Protocol': '-',
    'Protocol/Retrieval':'-'
}




### Resident 1

In [ ]:
#MRI Sequences
#Tokenize sequences
def tokenize(text):
    tokens = [token.strip() for token in text.split(',')]
    return tokens

#Function to calculate token-based accuracy
def calculate_symmetric_token_based_accuracy(ground_truth, result):
    ground_truth_tokens = tokenize(ground_truth)
    result_tokens = tokenize(result)
    
    ground_truth_counts = Counter(ground_truth_tokens)
    result_counts = Counter(result_tokens)
    
    # Calculate the number of matching tokens considering repetitions
    matching_tokens_gt_to_res = sum(min(ground_truth_counts[token], result_counts[token]) for token in ground_truth_counts)
    matching_tokens_res_to_gt = sum(min(ground_truth_counts[token], result_counts[token]) for token in result_counts)
    
    # Total tokens in ground truth and result (considering repetitions)
    total_tokens_gt = sum(ground_truth_counts.values())
    total_tokens_res = sum(result_counts.values())
    
    # Calculate accuracy from ground truth to result
    accuracy_gt_to_res = matching_tokens_gt_to_res / total_tokens_gt if total_tokens_gt > 0 else 0
    
    # Calculate accuracy from result to ground truth
    accuracy_res_to_gt = matching_tokens_res_to_gt / total_tokens_res if total_tokens_res > 0 else 0
    
    # Symmetric accuracy: average of both directions
    symmetric_accuracy = (accuracy_gt_to_res + accuracy_res_to_gt) / 2
    
    return symmetric_accuracy

# Example usage with your provided data:
ground_truths = df_GT['Sequenzen']
results = df_resident1['Sequenzen'] 

accuracies_res1 = []
for gt, res in zip(ground_truths, results):
    accuracy = calculate_symmetric_token_based_accuracy(gt, res)
    accuracies_res1.append(accuracy)

average_accuracy_seq = sum(accuracies_res1) / len(accuracies_res1)

# Cohen's Kappa calculation
# Create agreement labels: 1 if they agree (accuracy == 1), else 0
agreement_labels_ground_truth = []
agreement_labels_result = []

for gt, res in zip(ground_truths, results):
    # Tokenize and sort to compare token-based categories
    tokenized_gt = tokenize(gt)
    tokenized_res = tokenize(res)
    
    # Label agreement: 1 if identical tokens, else 0
    agreement_labels_ground_truth.append(' '.join(sorted(tokenized_gt)))
    agreement_labels_result.append(' '.join(sorted(tokenized_res)))

# Calculate Cohen's Kappa
kappa_score_seq = cohen_kappa_score(agreement_labels_ground_truth, agreement_labels_result)

# Bootstrap function to calculate confidence intervals
def bootstrap_confidence_interval(data, num_samples=10000, confidence_level=0.95):
    # Resample with replacement and calculate means
    sample_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(num_samples)]

    # Calculate the percentiles for the given confidence level
    lower_bound = np.percentile(sample_means, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(sample_means, (1 + confidence_level) / 2 * 100)

    return lower_bound, upper_bound
confidence_interval_seq = bootstrap_confidence_interval(accuracies_res1)

In [ ]:
#Contrast Medium Administration
# Comparing the two columns and calculating accuracy
df_equal_res1 = df_resident1['Kontrastmittelgabe'].astype(bool) == df_GT['Kontrastmittelgabe'].astype(bool)

# Calculating the number of correct predictions (where the comparison is True)
correct_predictions = df_equal_res1.sum()

# Calculating accuracy
average_accuracy_cm = correct_predictions / len(df_equal_res1) * 100

# Set the number of bootstrap samples
n_iterations = 10000
bootstrap_accuracies = []

# Number of total samples
n = len(df_equal_res1)

# Perform bootstrapping
for i in range(n_iterations):
    # Sample with replacement
    bootstrap_sample = np.random.choice(df_equal_res1, size=n, replace=True)
    
    # Calculate accuracy for this bootstrap sample
    accuracies = np.sum(bootstrap_sample) / len(bootstrap_sample) * 100
    bootstrap_accuracies.append(accuracies)

# Convert to a numpy array for easier manipulation
bootstrap_accuracies = np.array(bootstrap_accuracies)

# Calculate the 95% confidence interval
confidence_interval_cm = np.percentile(bootstrap_accuracies, [2.5, 97.5])

# Calculate Cohen's Kappa
kappa_score_cm = cohen_kappa_score(df_resident1['Kontrastmittelgabe'].astype(bool), df_GT['Kontrastmittelgabe'].astype(bool))

# New results to add
df_results.loc['Resident 1'] = {
    'Confidenz Interval Sequences': (confidence_interval_seq[0] * 100, confidence_interval_seq[1] * 100),
    'Average Accuracy Sequences': average_accuracy_seq * 100,
    'Kappa Score Sequences':kappa_score_seq,
    'Confidenz Interval Contrastmedium': (confidence_interval_cm[0], confidence_interval_cm[1]),
    'Average Accuracy Contrastmedium': average_accuracy_cm,
    'Kappa Score Contrastmedium': kappa_score_cm,
    'Number of Correct Retrieval': '-',
    'Number of Correct Protocol': '-',
    'Protocol/Retrieval':'-'
}


### Resident 2

In [ ]:
#MRI Sequences
#Tokenize sequences
def tokenize(text):
    tokens = [token.strip() for token in text.split(',')]
    return tokens

#Function to calculate token-based accuracy
def calculate_symmetric_token_based_accuracy(ground_truth, result):
    ground_truth_tokens = tokenize(ground_truth)
    result_tokens = tokenize(result)
    
    ground_truth_counts = Counter(ground_truth_tokens)
    result_counts = Counter(result_tokens)
    
    # Calculate the number of matching tokens considering repetitions
    matching_tokens_gt_to_res = sum(min(ground_truth_counts[token], result_counts[token]) for token in ground_truth_counts)
    matching_tokens_res_to_gt = sum(min(ground_truth_counts[token], result_counts[token]) for token in result_counts)
    
    # Total tokens in ground truth and result (considering repetitions)
    total_tokens_gt = sum(ground_truth_counts.values())
    total_tokens_res = sum(result_counts.values())
    
    # Calculate accuracy from ground truth to result
    accuracy_gt_to_res = matching_tokens_gt_to_res / total_tokens_gt if total_tokens_gt > 0 else 0
    
    # Calculate accuracy from result to ground truth
    accuracy_res_to_gt = matching_tokens_res_to_gt / total_tokens_res if total_tokens_res > 0 else 0
    
    # Symmetric accuracy: average of both directions
    symmetric_accuracy = (accuracy_gt_to_res + accuracy_res_to_gt) / 2
    
    return symmetric_accuracy

# Example usage with your provided data:
ground_truths = df_GT['Sequenzen']
results = df_resident2['Sequenzen'] 

accuracies_res2 = []
for gt, res in zip(ground_truths, results):
    accuracy = calculate_symmetric_token_based_accuracy(gt, res)
    accuracies_res2.append(accuracy)

average_accuracy_seq = sum(accuracies_res2) / len(accuracies_res2)

# Cohen's Kappa calculation
# Create agreement labels: 1 if they agree (accuracy == 1), else 0
agreement_labels_ground_truth = []
agreement_labels_result = []

for gt, res in zip(ground_truths, results):
    # Tokenize and sort to compare token-based categories
    tokenized_gt = tokenize(gt)
    tokenized_res = tokenize(res)
    
    # Label agreement: 1 if identical tokens, else 0
    agreement_labels_ground_truth.append(' '.join(sorted(tokenized_gt)))
    agreement_labels_result.append(' '.join(sorted(tokenized_res)))

# Calculate Cohen's Kappa
kappa_score_seq = cohen_kappa_score(agreement_labels_ground_truth, agreement_labels_result)

# Bootstrap function to calculate confidence intervals
def bootstrap_confidence_interval(data, num_samples=10000, confidence_level=0.95):
    # Resample with replacement and calculate means
    sample_means = [np.mean(np.random.choice(data, size=len(data), replace=True)) for _ in range(num_samples)]

    # Calculate the percentiles for the given confidence level
    lower_bound = np.percentile(sample_means, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(sample_means, (1 + confidence_level) / 2 * 100)

    return lower_bound, upper_bound
confidence_interval_seq = bootstrap_confidence_interval(accuracies_res2)

In [ ]:
#Contrast Medium Administration
# Comparing the two columns and calculating accuracy
df_equal_res2 = df_resident2['Kontrastmittelgabe'].astype(bool) == df_GT['Kontrastmittelgabe'].astype(bool)

# Calculating the number of correct predictions (where the comparison is True)
correct_predictions = df_equal_res2.sum()

# Calculating accuracy
average_accuracy_cm = correct_predictions / len(df_equal_res2) * 100

# Set the number of bootstrap samples
n_iterations = 10000
bootstrap_accuracies = []

# Number of total samples
n = len(df_equal_res2)

# Perform bootstrapping
for i in range(n_iterations):
    # Sample with replacement
    bootstrap_sample = np.random.choice(df_equal_res2, size=n, replace=True)
    
    # Calculate accuracy for this bootstrap sample
    accuracies = np.sum(bootstrap_sample) / len(bootstrap_sample) * 100
    bootstrap_accuracies.append(accuracies)

# Convert to a numpy array for easier manipulation
bootstrap_accuracies = np.array(bootstrap_accuracies)

# Calculate the 95% confidence interval
confidence_interval_cm = np.percentile(bootstrap_accuracies, [2.5, 97.5])

# Calculate Cohen's Kappa
kappa_score_cm = cohen_kappa_score(df_resident2['Kontrastmittelgabe'].astype(bool), df_GT['Kontrastmittelgabe'].astype(bool))

# New results to add
df_results.loc['Resident 2'] = {
    'Confidenz Interval Sequences': (confidence_interval_seq[0] * 100, confidence_interval_seq[1] * 100),
    'Average Accuracy Sequences': average_accuracy_seq * 100,
    'Kappa Score Sequences':kappa_score_seq,
    'Confidenz Interval Contrastmedium': (confidence_interval_cm[0], confidence_interval_cm[1]),
    'Average Accuracy Contrastmedium': average_accuracy_cm,
    'Kappa Score Contrastmedium': kappa_score_cm,
    'Number of Correct Retrieval': '-',
    'Number of Correct Protocol': '-',
    'Protocol/Retrieval':'-'
}

In [ ]:
df_results.to_csv(path_or_buf='path/results.csv', sep=';')

### Radiologist mean

In [ ]:
# Sequences 
all_accuracies_seq = (accuracies_res1 + accuracies_res2 + accuracies_rad1 + accuracies_rad2) 
mean_accuracy_seq = np.mean(all_accuracies_seq)
ci_seq = bootstrap_confidence_interval(all_accuracies_seq)
ci_seq_percent = (ci_seq[0] * 100, ci_seq[1] * 100)


# Contrast Media
all_cm = np.concatenate([df_equal_res1, df_equal_res2, df_equal_rad1, df_equal_rad2])
mean_accuracy_cm = np.mean(all_cm) * 100
bootstrap_accs = [np.mean(np.random.choice(all_cm, size=len(all_cm), replace=True)) * 100 
                  for _ in range(10000)]
ci_cm = np.percentile(bootstrap_accs, [2.5, 97.5])


df_rad_mean = pd.DataFrame([{
    "Radiologist": "All Radiologists (mean)",
    "Average Accuracy Sequences": mean_accuracy_seq * 100,
    "Confidenz Interval Sequences": ci_seq_percent,
    "Average Accuracy Contrastmedium": mean_accuracy_cm,
    "Confidenz Interval Contrastmedium": ci_cm 
    }])

df_rad_mean.to_csv('/path/results', index=False, sep=";")

## COMPARISON BETWEEN MODEL RESULTS

### LLAMA Sequences non-RAG to RAG

In [ ]:
model_a_accuracies = accuracies_OS_woRAG
model_b_accuracies = accuracies_OS_RAG

# Perform Wilcoxon signed-rank test
stat, p_value = wilcoxon(model_a_accuracies, model_b_accuracies)

# Output the results
print(f"Wilcoxon test statistic: {stat}")
print(f"P-value: {p_value}")

comparison_results = [{
    'P-Value': p_value
}]
index = ['Llama 3.1 RAG vs. Llama 4 Maverick RAG Sequences']

df_comparison_results = pd.DataFrame(comparison_results, index=index)

### LLAMA Contrastmedium non-RAG to RAG

In [ ]:
# Comparing the two columns 
df_equal_OS_woRAG = df_OS_woRAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 
df_equal_OS_RAG = df_OS_RAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 

# Lists of boolean predictions from two LLMs
llm1_predictions = df_equal_OS_RAG
llm2_predictions = df_equal_OS_woRAG

# Initialize counts for the 2x2 contingency table
a = b = c = d = 0

# Compare each prediction pair and update contingency table counts
for pred1, pred2 in zip(llm1_predictions, llm2_predictions):
    if pred1 == True and pred2 == True:
        a += 1
    elif pred1 == True and pred2 == False:
        b += 1
    elif pred1 == False and pred2 == True:
        c += 1
    elif pred1 == False and pred2 == False:
        d += 1

# Create the contingency table
table = np.array([[a, b],
                  [c, d]])

# Perform the McNemar test
result = mcnemar(table, exact=True)  # Set exact=True for small samples

# Output the test statistic and p-value
print(f'Contingency table: \n{table}')
print(f'Chi-squared: {result.statistic}, p-value: {result.pvalue}')
p_value = (result.pvalue)

df_comparison_results.loc['Llama 3.1 RAG vs. Llama 4 Maverick RAG Contrast Media']= {'P-Value':p_value}



### GPT Sequences non-RAG to RAG

In [ ]:
model_a_accuracies = accuracies_woRAG
model_b_accuracies = accuracies_RAG

# Perform Wilcoxon signed-rank test
stat, p_value = wilcoxon(model_a_accuracies, model_b_accuracies)

# Output the results
print(f"Wilcoxon test statistic: {stat}")
print(f"P-value: {p_value}")

df_comparison_results.loc['Claude RAG vs. GPT-5.2 RAG Sequences'] = {'P-Value':p_value}



### GPT Contrastmedium non-RAG to RAG

In [ ]:
# Comparing the two columns 
df_equal_woRAG = df_woRAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 
df_equal_RAG = df_RAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 

# Lists of boolean predictions from two LLMs
llm1_predictions = df_equal_woRAG
llm2_predictions = df_equal_RAG

# Initialize counts for the 2x2 contingency table
a = b = c = d = 0

# Compare each prediction pair and update contingency table counts
for pred1, pred2 in zip(llm1_predictions, llm2_predictions):
    if pred1 == True and pred2 == True:
        a += 1
    elif pred1 == True and pred2 == False:
        b += 1
    elif pred1 == False and pred2 == True:
        c += 1
    elif pred1 == False and pred2 == False:
        d += 1

# Create the contingency table
table = np.array([[a, b],
                  [c, d]])

# Perform the McNemar test
result = mcnemar(table, exact=False)  # Set exact=True for small samples

# Output the test statistic and p-value
print(f'Contingency table: \n{table}')
print(f'Chi-squared: {result.statistic}, p-value: {result.pvalue}')
p_value = result.pvalue

df_comparison_results.loc['Claude RAG vs. GPT-5.2 RAG Contrast Media']= {'P-Value':p_value}


### Claude Sequences non-RAG to RAG

In [ ]:
model_a_accuracies = accuracies_claude_woRAG
model_b_accuracies = accuracies_claude_RAG

# Perform Wilcoxon signed-rank test
stat, p_value = wilcoxon(model_a_accuracies, model_b_accuracies)

# Output the results
print(f"Wilcoxon test statistic: {stat}")
print(f"P-value: {p_value}")

df_comparison_results.loc['Claude Sequences non-RAG to RAG'] = {'P-Value':p_value}

### Claude Contrastmedium non-RAG to RAG

In [ ]:
# Comparing the two columns 
df_equal_woRAG = df_claude_woRAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 
df_equal_RAG = df_claude_RAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 

# Lists of boolean predictions from two LLMs
llm1_predictions = df_equal_woRAG
llm2_predictions = df_equal_RAG

# Initialize counts for the 2x2 contingency table
a = b = c = d = 0

# Compare each prediction pair and update contingency table counts
for pred1, pred2 in zip(llm1_predictions, llm2_predictions):
    if pred1 == True and pred2 == True:
        a += 1
    elif pred1 == True and pred2 == False:
        b += 1
    elif pred1 == False and pred2 == True:
        c += 1
    elif pred1 == False and pred2 == False:
        d += 1

# Create the contingency table
table = np.array([[a, b],
                  [c, d]])

# Perform the McNemar test
result = mcnemar(table, exact=False)  # Set exact=True for small samples

# Output the test statistic and p-value
print(f'Contingency table: \n{table}')
print(f'Chi-squared: {result.statistic}, p-value: {result.pvalue}')
p_value = result.pvalue

df_comparison_results.loc['Claude Contrastmedium non-RAG to RAG']= {'P-Value':p_value}

In [ ]:
df_comparison_results.to_csv(path_or_buf='/path/results.csv', sep=';')

### Radiologists to GPT with RAG

In [ ]:
mean_accuracies = []

# Loop through the values in all accuracy lists simultaneously
for res2, res1, rad1, rad2 in zip(accuracies_res2, accuracies_res1, accuracies_rad1, accuracies_rad2):
    accuracy = (res2 + res1 + rad1 + rad2) / 4 
    mean_accuracies.append(accuracy) 

mean_accuracies_CM = []

# Loop through the values in all accuracy lists simultaneously
for res2, res1, rad1, rad2 in zip(df_equal_rad1, df_equal_rad2, df_equal_res1, df_equal_res2):
    accuracy = (res2 + res1 + rad1 + rad2) / 4  # Calculate the average accuracy
    mean_accuracies_CM.append(accuracy)  # Append the result to mean_accuracy list

# Calculate standard deviation for both sets of accuracies
sd_mean_accuracy = np.std(mean_accuracies)
sd_mean_accuracy_CM = np.std(mean_accuracies_CM)

#Calculate mean accuracy 
mean_accuracy = sum(mean_accuracies)/len(mean_accuracies)
mean_accuracy_CM = sum(mean_accuracies_CM)/len(mean_accuracies_CM)

In [ ]:
# MRI sequences — Wilcoxon
stat, p_value = wilcoxon(mean_accuracies, accuracies_RAG)
print(f"Sequences: Wilcoxon stat={stat}, p={p_value:.6f}")
df_comparison_results.loc['Radiologists to GPT RAG Sequences'] = {'P-Value': p_value}

# Contrast media — Wilcoxon
llm_correct_cm = (df_RAG['Vorhersage Kontrastmittelgabe'].astype(int) == df_GT['Kontrastmittelgabe'].astype(int)).astype(float)

stat, p_value = wilcoxon(mean_accuracies_CM, llm_correct_cm)
print(f"Contrast Media: Wilcoxon stat={stat}, p={p_value:.6f}")
df_comparison_results.loc['Radiologists to GPT RAG Contrastmedium'] = {'P-Value': p_value}

### Radiologists to LLama with RAG

In [ ]:
# MRI sequences — Wilcoxon
stat, p_value = wilcoxon(mean_accuracies, accuracies_OS_RAG)
print(f"Sequences: Wilcoxon stat={stat}, p={p_value:.6f}")
df_comparison_results.loc['Radiologists to Llama RAG Sequences'] = {'P-Value': p_value}

# Contrast media — Wilcoxon
llm_correct_cm = (df_OS_RAG['Vorhersage Kontrastmittelgabe'].astype(int) == df_GT['Kontrastmittelgabe'].astype(int)).astype(float)

stat, p_value = wilcoxon(mean_accuracies_CM, llm_correct_cm)
print(f"Contrast Media: Wilcoxon stat={stat}, p={p_value:.6f}")
df_comparison_results.loc['Radiologists to Llama RAG Contrastmedium'] = {'P-Value': p_value}

### Radiologists to Claude RAG

In [ ]:
# MRI sequences — Wilcoxon
stat, p_value = wilcoxon(mean_accuracies, accuracies_claude_RAG)
print(f"Sequences: Wilcoxon stat={stat}, p={p_value:.6f}")
df_comparison_results.loc['Radiologists to Claude RAG Sequences'] = {'P-Value': p_value}

# Contrast media — Wilcoxon
llm_correct_cm = (df_claude_RAG['Vorhersage Kontrastmittelgabe'].astype(int) == df_GT['Kontrastmittelgabe'].astype(int)).astype(float)

stat, p_value = wilcoxon(mean_accuracies_CM, llm_correct_cm)
print(f"Contrast Media: Wilcoxon stat={stat}, p={p_value:.6f}")
df_comparison_results.loc['Radiologists to Claude RAG Contrastmedium'] = {'P-Value': p_value}

In [ ]:
df_comparison_results.to_csv(path_or_buf='/path/results.csv', sep=';')

### LLAMA RAG to GPT RAG Sequences

In [ ]:
model_a_accuracies = accuracies_OS_RAG
model_b_accuracies = accuracies_RAG

# Perform Wilcoxon signed-rank test
stat, p_value = wilcoxon(model_a_accuracies, model_b_accuracies)

# Output the results
print(f"Wilcoxon test statistic: {stat}")
print(f"P-value: {p_value}")

df_comparison_results.loc['Llama vs GPT RAG Sequences']= {'P-Value':p_value}

### LLAMA RAG to GPT RAG Contrastmedium

In [ ]:
# Comparing the two columns 
df_equal_OS_RAG = df_OS_RAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 
df_equal_RAG = df_RAG['Vorhersage Kontrastmittelgabe'] == df_GT['Kontrastmittelgabe'] 

# Lists of boolean predictions from two LLMs
llm1_predictions = df_equal_OS_RAG
llm2_predictions = df_equal_RAG

# Initialize counts for the 2x2 contingency table
a = b = c = d = 0

# Compare each prediction pair and update contingency table counts
for pred1, pred2 in zip(llm1_predictions, llm2_predictions):
    if pred1 == True and pred2 == True:
        a += 1
    elif pred1 == True and pred2 == False:
        b += 1
    elif pred1 == False and pred2 == True:
        c += 1
    elif pred1 == False and pred2 == False:
        d += 1

# Create the contingency table
table = np.array([[a, b],
                  [c, d]])

# Perform the McNemar test
result = mcnemar(table, exact=True)  # Set exact=True for small samples

# Output the test statistic and p-value
print(f'Contingency table: \n{table}')
print(f'Chi-squared: {result.statistic}, p-value: {result.pvalue}')
p_value = result.pvalue


df_comparison_results.loc['Llama vs GPT RAG Contrastmedia']= {'P-Value':p_value}

### Evaluation Across Years

In [ ]:
# Sequence accuracy
acc_gpt4o_2024 = np.array(accuracies_woRAG)
acc_llama31_2024 = np.array(accuracies_OS_woRAG)
acc_gpt52_2026 = np.array(accuracies_RAG)
acc_llama4_2026 = np.array(accuracies_OS_RAG)

# CM accuracy
cm_gpt4o_2024 = df_equal_woRAG.astype(int).values
cm_llama31_2024 = df_equal_OS_woRAG.astype(int).values
cm_gpt52_2026 = df_equal_RAG.astype(int).values
cm_llama4_2026 = df_equal_OS_RAG.astype(int).values


# Sequence accuracy: Wilcoxon signed-rank on Δgap
gap_2024_seq  = acc_gpt4o_2024 - acc_llama31_2024
gap_2026_seq  = acc_gpt52_2026 - acc_llama4_2026
delta_gap_seq = gap_2024_seq - gap_2026_seq

stat_seq, p_seq = wilcoxon(delta_gap_seq, alternative='greater')

# Bootstrap 95% CI on mean Δgap 
rng = np.random.default_rng(42)
n_seq = len(delta_gap_seq)
boot_means_seq = np.array([
    rng.choice(delta_gap_seq, size=n_seq, replace=True).mean()
    for _ in range(10_000)
])
ci_lo_seq, ci_hi_seq = np.percentile(boot_means_seq, [2.5, 97.5])

# CM accuracy: Wilcoxon on Δgap 
gap_2024_cm  = cm_gpt4o_2024 - cm_llama31_2024
gap_2026_cm  = cm_gpt52_2026 - cm_llama4_2026
delta_gap_cm = gap_2024_cm - gap_2026_cm

try:
    stat_cm, p_cm = wilcoxon(delta_gap_cm, alternative='greater',
                             zero_method='wilcox')
except ValueError as e:
    stat_cm, p_cm = np.nan, np.nan
    print(f"Wilcoxon (CM) could not be computed: {e}")

# Bootstrap 95% CI on the difference of proportion gaps — preferred for binary data
n_cm = len(cm_gpt4o_2024)
boot_deltas_cm = []
for _ in range(10_000):
    idx = rng.integers(0, n_cm, size=n_cm)              
    g24 = cm_gpt4o_2024[idx].mean()  - cm_llama31_2024[idx].mean()
    g26 = cm_gpt52_2026[idx].mean()  - cm_llama4_2026[idx].mean()
    boot_deltas_cm.append(g24 - g26)
boot_deltas_cm = np.array(boot_deltas_cm)

observed_delta_cm   = gap_2024_cm.mean() - gap_2026_cm.mean()
ci_lo_cm, ci_hi_cm  = np.percentile(boot_deltas_cm, [2.5, 97.5])
p_boot_cm           = np.mean(boot_deltas_cm <= 0)      


#Create dataframe
comparison_results = [{
    'Median Gap 2024 (pp)': np.median(gap_2024_seq) * 100,
    'Median Gap 2026 (pp)': np.median(gap_2026_seq) * 100,
    'Median Δgap (pp)':     np.median(delta_gap_seq) * 100,
    'Bootstrap CI Lower (pp)': ci_lo_seq * 100,
    'Bootstrap CI Upper (pp)': ci_hi_seq * 100,
    'Test Statistic': stat_seq,
    'P-Value': p_seq,
}]
index = ['H3: Proprietary–Open gap narrowing (Sequences)']

df_comparison_results = pd.DataFrame(comparison_results, index=index)

df_comparison_results.loc['Proprietary–Open gap narrowing (CM, bootstrap)'] = {
    'Median Gap 2024 (pp)': gap_2024_cm.mean() * 100,
    'Median Gap 2026 (pp)': gap_2026_cm.mean() * 100,
    'Median Δgap (pp)':     observed_delta_cm * 100,
    'Bootstrap CI Lower (pp)': ci_lo_cm * 100,
    'Bootstrap CI Upper (pp)': ci_hi_cm * 100,
    'Test Statistic': np.nan,
    'P-Value': p_boot_cm,
}

df_comparison_results.loc['Proprietary–Open gap narrowing (CM, Wilcoxon)'] = {
    'Median Gap 2024 (pp)': gap_2024_cm.mean() * 100,
    'Median Gap 2026 (pp)': gap_2026_cm.mean() * 100,
    'Median Δgap (pp)':     observed_delta_cm * 100,
    'Bootstrap CI Lower (pp)': np.nan,
    'Bootstrap CI Upper (pp)': np.nan,
    'Test Statistic': stat_cm,
    'P-Value': p_cm,
}

df_comparison_results.to_csv(path_or_buf='/path/results.csv', sep=';')

# FIGURES

In [ ]:
import os
import re
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.decomposition import PCA

## VEKTOR STORE

In [ ]:
# Load the OpenAI API key 
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
 
# Extract the protocol label 
def protocol_from_doc(doc) -> str:
    for line in (doc.page_content or "").splitlines():
        line = line.strip()
        if line:
            return line
    return "UNKNOWN"
 
 
# Map a protocol string to a region-qualified group label
def map_protocol_to_group(proto: str) -> str:
    p = (proto or "").lower()
 
    # Robust region detection
    if re.search(r"\b(wirbelsäule|ws|hws|bws|lws|sakrum)\b", p):
        region = "Wirbelsäule"
    elif re.search(r"\b(kopf|schädel|hirn|cerebrum|neuro)\b", p):
        region = "Kopf"
    else:
        region = "Unbekannt"
 
 
    if region == "Kopf" and ("routine" in p) and ("trauma" in p):
        cat = "Routine"
    else:
        if "vaskulär" in p:
            cat = "Vaskulär"
        elif "tumor" in p:
            cat = "Tumor"
        elif "entzündung" in p or re.search(r"\bms\b", p):
            cat = "Entzündung"
        elif "neurodegeneration" in p:
            cat = "Neurodegeneration"
        elif "spezielle regionen" in p or "vordere schädelbasis" in p:
            cat = "Spezielle Regionen"
        elif "degenerativ" in p:
            cat = "Degenerativ"
        elif "trauma" in p:
            cat = "Trauma"
        elif re.search(r"\b(routine|standard|basis)\b", p):
            cat = "Routine"
        else:
            cat = "Andere"
 
    # Restrict to the allowed classes
    if region == "Kopf":
        allowed = {"Routine", "Vaskulär", "Tumor", "Entzündung", "Neurodegeneration", "Spezielle Regionen"}
        return f"{cat} (Kopf)" if cat in allowed else "Andere (Kopf)"
    if region == "Wirbelsäule":
        allowed = {"Routine", "Vaskulär", "Tumor", "Entzündung", "Degenerativ", "Trauma"}
        return f"{cat} (Wirbelsäule)" if cat in allowed else "Andere (Wirbelsäule)"
    return "Andere (Unbekannt)"
 
 
# L2-normalize embedding rows 
def l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)
 
 
# Derive a group label for every chunk
protocols = [protocol_from_doc(d) for d in docs]
groups = [map_protocol_to_group(p) for p in protocols]
 
# Fixed legend/color order
preferred_order = [
    "Routine (Kopf)",
    "Vaskulär (Kopf)",
    "Tumor (Kopf)",
    "Entzündung (Kopf)",
    "Neurodegeneration (Kopf)",
    "Spezielle Regionen (Kopf)",
    "Routine (Wirbelsäule)",
    "Vaskulär (Wirbelsäule)",
    "Tumor (Wirbelsäule)",
    "Entzündung (Wirbelsäule)",
    "Degenerativ (Wirbelsäule)",
    "Trauma (Wirbelsäule)",
    "Andere (Kopf)",
    "Andere (Wirbelsäule)",
    "Andere (Unbekannt)",
]
groups_order = [g for g in preferred_order if g in set(groups)]
 
# Model A vectors
embedding_ada = OpenAIEmbeddings(openai_api_key=openai_api_key, model="text-embedding-ada-002")
vectors_a = l2_normalize(np.array(embedding_ada.embed_documents(texts)))
 
# Model B vectors
embedding_large = OpenAIEmbeddings(openai_api_key=openai_api_key, model="text-embedding-3-large")
vectors_b = l2_normalize(np.array(embedding_large.embed_documents([d.page_content for d in docs])))
 
# Reduce both models to three principal components
pca3_a = PCA(n_components=3, random_state=42)
X3_a = pca3_a.fit_transform(vectors_a)
pca3_b = PCA(n_components=3, random_state=42)
X3_b = pca3_b.fit_transform(vectors_b)
 
# Assemble per-model DataFrames for plotting
df3_a = pd.DataFrame({"PC1": X3_a[:, 0], "PC2": X3_a[:, 1], "PC3": X3_a[:, 2], "Gruppe": groups})
df3_b = pd.DataFrame({"PC1": X3_b[:, 0], "PC2": X3_b[:, 1], "PC3": X3_b[:, 2], "Gruppe": groups})
 
# Color palette
palette = sns.color_palette("tab20", n_colors=len(groups_order))
color_map = {g: palette[i] for i, g in enumerate(groups_order)}
 
 
# Style
def set_paper_style():
    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 15,
        "axes.labelsize": 15,
        "legend.fontsize": 15,
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,  
        "ps.fonttype": 42,
    })
 
 
# Style 3D axis
def style_3d_paper(ax, lim, elev=18, azim=-60, label_pad=7):
    # Symmetric limits 
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(-lim, lim)
 
    # Projection / aspect / view
    ax.set_proj_type("ortho")
    ax.set_box_aspect([1, 1, 1])
    ax.view_init(elev=elev, azim=azim)
 
    # Grid and panes
    ax.grid(False)
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        try:
            axis.pane.fill = False
            axis.pane.set_edgecolor((1, 1, 1, 0))
        except Exception:
            pass
 
    # Axis lines 
    ax.plot([-lim, lim], [0, 0], [0, 0], color="black", linewidth=1.0, alpha=0.85)
    ax.plot([0, 0], [-lim, lim], [0, 0], color="black", linewidth=1.0, alpha=0.85)
    ax.plot([0, 0], [0, 0], [-lim, lim], color="black", linewidth=1.0, alpha=0.85)
 
    # Labels & Ticks
    ax.set_xlabel("PC1", labelpad=label_pad)
    ax.set_ylabel("PC2", labelpad=label_pad)
    ax.set_zlabel("PC3", labelpad=label_pad)
    ax.xaxis.set_tick_params(labelsize=10)
    ax.yaxis.set_tick_params(labelsize=10)
    ax.zaxis.set_tick_params(labelsize=10)
 
 
# Scatter the PCA points, colored by protocol group
def scatter_by_group_paper(ax, df3, groups_order, color_map, s=50, alpha=0.95):
    for g in groups_order:
        d = df3[df3["Gruppe"] == g]
        if d.empty:
            continue
        ax.scatter(
            d["PC1"], d["PC2"], d["PC3"],
            s=s,
            c=[color_map[g]],
            alpha=alpha,
            edgecolors="none",  
            depthshade=False,   
        )
 
 
# Side-by-side 3D PCA figure (Model A vs. Model B)
set_paper_style()
lim = 0.3 
 
fig = plt.figure(figsize=(18, 7))
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
 
scatter_by_group_paper(ax1, df3_a, groups_order, color_map)
style_3d_paper(ax1, lim, elev=18, azim=-60, label_pad=4)
 
scatter_by_group_paper(ax2, df3_b, groups_order, color_map)
style_3d_paper(ax2, lim, elev=18, azim=-60, label_pad=4)
 
# Titles
title1 = r"$\bf{3D\ PCA\ -\ text\!-\!embedding\!-\!ada\!-\!002}$"
sub1 = (f"PC1={pca3_a.explained_variance_ratio_[0]*100:.1f}%, "
        f"PC2={pca3_a.explained_variance_ratio_[1]*100:.1f}%, "
        f"PC3={pca3_a.explained_variance_ratio_[2]*100:.1f}%")
 
title2 = r"$\bf{3D\ PCA\ -\ text\!-\!embedding\!-\!3\!-\!large}$"
sub2 = (f"PC1={pca3_b.explained_variance_ratio_[0]*100:.1f}%, "
        f"PC2={pca3_b.explained_variance_ratio_[1]*100:.1f}%, "
        f"PC3={pca3_b.explained_variance_ratio_[2]*100:.1f}%")
 
ax1.text2D(0.5, 1.02, title1, transform=ax1.transAxes, ha="center", va="bottom", fontsize=13)
ax1.text2D(0.5, 0.98, sub1, transform=ax1.transAxes, ha="center", va="bottom", fontsize=10)
ax2.text2D(0.5, 1.02, title2, transform=ax2.transAxes, ha="center", va="bottom", fontsize=13)
ax2.text2D(0.5, 0.98, sub2, transform=ax2.transAxes, ha="center", va="bottom", fontsize=10)
 
# Legend
legend_handles = [
    plt.Line2D([0], [0], marker="o", linestyle="",
               markersize=10,
               markerfacecolor=color_map[g],
               markeredgecolor="none",
               label=g)
    for g in groups_order
]
legend = fig.legend(
    handles=legend_handles,
    labels=groups_order,
    title="Protokollgruppe",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=True,
)
legend.get_title().set_fontweight("bold")
 
plt.show()

## RETRIEVAL

In [ ]:
#Load data
df_retrieval = pd.read_csv('/path/retrieval results.csv', sep=';')

#Reshape data
df = df_retrieval.rename(columns={"Unnamed: 0": "k"}).copy()
df_ci = pd.concat([
    df[["k", "ada-002 Hit Rate", "ada-002 Lower (95%)", "ada-002 Upper (95%)"]]
      .rename(columns={
          "ada-002 Hit Rate": "Score",
          "ada-002 Lower (95%)": "ci_low",
          "ada-002 Upper (95%)": "ci_high"
      })
      .assign(Modell="Text-embedding-ada-002"),

    df[["k", "3-Large Hit Rate", "3-Large CI Lower (95%)", "3-Large CI Upper (95%)"]]
      .rename(columns={
          "3-Large Hit Rate": "Score",
          "3-Large CI Lower (95%)": "ci_low",
          "3-Large CI Upper (95%)": "ci_high"
      })
      .assign(Modell="Text-embedding-3-large")
], ignore_index=True)

df_ci["Score"] = df_ci["Score"] * 100
df_ci["ci_low"] = df_ci["ci_low"] * 100
df_ci["ci_high"] = df_ci["ci_high"] * 100

# Style
sns.set_theme(style="whitegrid", context="talk")
fig, ax = plt.subplots(figsize=(10, 6))

palette = {
    "Text-embedding-ada-002": "#F68E6F",
    "Text-embedding-3-large": "#F75A3B"
}

sns.lineplot(
    data=df_ci,
    x="k", y="Score",
    hue="Modell",
    palette=palette,
    linewidth=3,
    ax=ax
)

# CI-Lines
for modell, sub in df_ci.groupby("Modell"):
    sub = sub.sort_values("k")
    ax.fill_between(
        sub["k"].to_numpy(),
        sub["ci_low"].to_numpy(),
        sub["ci_high"].to_numpy(),
        alpha=0.25,
        color=palette.get(modell)
    )

# Axis & Ticks
k_ticks = sorted(df_ci["k"].dropna().unique())
ax.set_xticks(k_ticks)
ax.set_xticklabels([str(int(k)) if float(k).is_integer() else str(k) for k in k_ticks])
ax.set_ylim(50, 100)
ax.set_xlim(2, 10)


ax.set_xlabel("top-$k$ values")
ax.set_ylabel("Correct Retrieval (%)")
legend = ax.legend(title="Embedding Model")
legend.get_title().set_fontweight("bold")
fig.tight_layout()
plt.show()

In [ ]:
#Load data
df_retrieval_error = pd.read_csv('/path/retrieval error.csv', sep=';')

# Style 
sns.set_theme(style="whitegrid", context="talk")

# Color palette
colors = ["#FFCCBD", "#FFA78D", "#FC8561", "#F75A3B", "#DF4E32"]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie Chart: ada-002 
values_ada = df_retrieval_error["Text-embedding-ada-002"]
labels_ada = df_retrieval_error["Type of Error"]

wedges1, texts1 = axes[0].pie(
    values_ada,
    labels=None,
    autopct=None,
    colors=colors,
    startangle=140,
    pctdistance=0.75,
    wedgeprops=dict(edgecolor="white", linewidth=2)
)

# Absolute values 
for i, (wedge, val) in enumerate(zip(wedges1, values_ada)):
    angle = (wedge.theta1 + wedge.theta2) / 2
    x = 0.75 * np.cos(np.radians(angle))
    y = 0.75 * np.sin(np.radians(angle))
    axes[0].text(x, y, str(val), ha="center", va="center",fontsize=11, fontweight="bold")
    
axes[0].set_title("Text-embedding-ada-002 (n=20.5)", fontweight="bold", pad=20)

# Pie Chart: 3-large 
mask = df_retrieval_error["Text-embedding-3-large"].notna()
values_large = df_retrieval_error.loc[mask, "Text-embedding-3-large"]
labels_large = df_retrieval_error.loc[mask, "Type of Error"]

wedges2, texts2 = axes[1].pie(
    values_large,
    labels=None,
    autopct=None,
    colors=colors[:mask.sum()],
    startangle=140,
    pctdistance=0.75,
    wedgeprops=dict(edgecolor="white", linewidth=2)
)

# Absolute values 
for i, (wedge, val) in enumerate(zip(wedges2, values_large)):
    if val == 0:continue
    angle = (wedge.theta1 + wedge.theta2) / 2
    x = 0.75 * np.cos(np.radians(angle))
    y = 0.75 * np.sin(np.radians(angle))
    axes[1].text(x, y, str(val), ha="center", va="center",fontsize=11, fontweight="bold")

axes[1].set_title("Text-embedding-3-large (n=8)", fontweight="bold", pad=20)

# Legend
fig.legend(
    wedges1, labels_ada,
    title="Type of Error",
    title_fontproperties={"weight": "bold"},
    loc="lower center",
    ncol=3,
    bbox_to_anchor=(0.5, -0.05),
    fontsize=13
)

fig.tight_layout()
plt.show()

## SEQUENCE EVALUATION

In [ ]:
# Load data 
df = pd.read_csv('/path/results sequences.csv',sep=';')

# Column names
model_col = "Unnamed: 0"
redundant_col = "Mean Redundant Sequences"
missing_col = "Mean Missing Sequences"

# Ensure numeric
df[redundant_col] = pd.to_numeric(df[redundant_col], errors="coerce")
df[missing_col] = pd.to_numeric(df[missing_col], errors="coerce")

# Model order 
llm_order = ["Claude Opus 4.6", "Llama 3.1 405B", "Llama 4 Maverick", "GPT-4o", "GPT-5.2"]

# Extract paired (without/with RAG) 
def extract_series(metric_col):
    no_rag, with_rag = [], []
    for model in llm_order:
        val_no = df.loc[df[model_col] == model, metric_col].values
        no_rag.append(val_no[0] if len(val_no) > 0 else np.nan)

        val_with = df.loc[df[model_col] == f"{model} RAG", metric_col].values
        with_rag.append(val_with[0] if len(val_with) > 0 else np.nan)
    return np.array(no_rag), np.array(with_rag)

missing_no, missing_with = extract_series(missing_col)
redundant_no, redundant_with = extract_series(redundant_col)

# Radar plot helper 
def radar_plot(ax, categories, values_no, values_with, title, rmin=0, rmax=12):
    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False)

    angles = np.concatenate([angles, [angles[0]]])
    values_no = np.concatenate([values_no, [values_no[0]]])
    values_with = np.concatenate([values_with, [values_with[0]]])

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories)
    ax.tick_params(axis="y", colors="gray")  
    ax.set_ylim(rmin, rmax)

    # Without RAG
    ax.plot(angles, values_no, linewidth=2.5, color="#B9AD91", label="without RAG")
    ax.fill(angles, values_no, color="#B9AD91", alpha=0.2)

    # With RAG
    ax.plot(angles, values_with, linewidth=2.5, color="#F75A3B", label="with RAG")
    ax.fill(angles, values_with, color="#F75A3B", alpha=0.5)

    ax.set_title(title, pad=20, fontweight="bold")
    ax.grid(True, alpha=0.4)

fig, axes = plt.subplots(1, 2, figsize=(17, 9), subplot_kw=dict(polar=True))

radar_plot(axes[0], llm_order, missing_no, missing_with,
           "Missing Sequences", rmin=0, rmax=4)

radar_plot(axes[1], llm_order, redundant_no, redundant_with,
           "Redundant Sequences", rmin=0, rmax=12)

# Legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2)

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

## RAG vs. RADIOLOGISTS

In [ ]:
df = pd.read_csv('/path/results.csv',sep=';')

# Radiologen-Namen aus "Unnamed: 0"
df = df.rename(columns={"Unnamed: 0": "Name"})

# Falls pandas es als Index interpretiert hat:
if "Name" not in df.columns:
    df = df.reset_index().rename(columns={"index": "Name"})

# Moderner Stil
sns.set_theme(style="whitegrid", context="talk")

# ---- CI robust parsen ----
def parse_ci(s):
    nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", str(s))
    return float(nums[0]), float(nums[1])

df[["Seq_lower", "Seq_upper"]] = df["Confidenz Interval Sequences"].apply(
    lambda x: pd.Series(parse_ci(x))
)
df[["CM_lower", "CM_upper"]] = df["Confidenz Interval Contrastmedium"].apply(
    lambda x: pd.Series(parse_ci(x))
)

df["Seq_mean"] = pd.to_numeric(df["Average Accuracy Sequences"], errors="coerce")
df["CM_mean"]  = pd.to_numeric(df["Average Accuracy Contrastmedium"], errors="coerce")

# Asymmetrische Fehlerbalken
seq_xerr = np.vstack([df["Seq_mean"] - df["Seq_lower"],
                      df["Seq_upper"] - df["Seq_mean"]])

cm_xerr = np.vstack([df["CM_mean"] - df["CM_lower"],
                     df["CM_upper"] - df["CM_mean"]])

# ---- Kategorien (RAG / ohne RAG / Radiologen) ----
def classify(name: str) -> str:
    s = str(name).lower()

    # Radiologen immer zuerst prüfen
    if "radiolog" in s:
        return "Radiologists"
    
    # Modelle MIT RAG explizit gekennzeichnet
    if "rag" in s or "mit rag" in s:
        return "with RAG"
    
    # Alles andere sind Modelle OHNE RAG
    return "without RAG"

df["Group"] = df["Name"].apply(classify)

# ---- Farben ----
# Sequences (kräftiger)
seq_palette = {
    "without RAG": "#B9AD91",   # gelb
    "with RAG":    "#F75A3B",   # lila
    "Radiologists":"#4C72B0"    # blau (wie vorher)
}

# Contrast Media (heller, "so wie jetzt auch")
cm_palette = {
    "without RAG": "#D1C29D",   # helles, weiches Gelb
    "with RAG":    "#F68E6F",   # helles Pastell-Lila
    "Radiologists":"#A9C3E6"    # dein gewünschtes helles Blau
}

seq_colors = df["Group"].map(seq_palette).tolist()
cm_colors  = df["Group"].map(cm_palette).tolist()

# ---- Plot: Zwei Diagramme nebeneinander ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(19, 15), sharey=True)

y = np.arange(len(df))
h = 0.6

# Panel 1 – MRI Sequences
ax1.barh(y, df["Seq_mean"], height=h, color=seq_colors, alpha=0.9, zorder=3)
ax1.errorbar(df["Seq_mean"], y, xerr=seq_xerr, fmt="none",
             ecolor="0.3", elinewidth=1.5, capsize=3, capthick=1.5, zorder=4)
ax1.set_title(f"MRI Sequence Prediction", fontweight="bold", pad=15, fontsize=25)
ax1.set_xlabel("Accuracy (%)")
ax1.set_xlim(0, 100)
ax1.set_yticks(y)
ax1.set_yticklabels(df["Name"])

# Panel 2 – Contrast Media
ax2.barh(y, df["CM_mean"], height=h, color=cm_colors, alpha=0.9, zorder=3)
ax2.errorbar(df["CM_mean"], y, xerr=cm_xerr, fmt="none",
             ecolor="0.3", elinewidth=1.5, capsize=3, capthick=1.5, zorder=4)
ax2.set_title(f"Contrast Medium Prediction", fontweight="bold", pad=15, fontsize=25)
ax2.set_xlabel("Accuracy (%)")
ax2.set_xlim(0, 100)

# ---- Gemeinsames Styling ----
sns.despine(left=True, bottom=True)

# ---- Legende (gemeinsam) ----
legend_handles = [
    Patch(facecolor=seq_palette["without RAG"],    edgecolor="none", label="Models without RAG"),
    Patch(facecolor=seq_palette["with RAG"],    edgecolor="none", label="Models with RAG"),
    Patch(facecolor=seq_palette["Radiologists"],edgecolor="none", label="Radiologists"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=3, fontsize=20, bbox_to_anchor=(0.5,-0.05), frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# Load data
df = pd.read_csv('/path/results sequences.csv',sep=';')
df = df.rename(columns={"Unnamed: 0": "Name"})

sns.set_theme(style="whitegrid", context="talk")

# Parse confidence interval 
def parse_ci(s):
    nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", str(s))
    return float(nums[0]), float(nums[1])

# Split the interval strings into numeric bound columns
df[["Seq_lower", "Seq_upper"]] = df["Confidenz Interval Sequences"].apply(
    lambda x: pd.Series(parse_ci(x))
)
df[["CM_lower", "CM_upper"]] = df["Confidenz Interval Contrastmedium"].apply(
    lambda x: pd.Series(parse_ci(x))
)

# Convert to numeric values
df["Seq_mean"] = pd.to_numeric(df["Average Accuracy Sequences"], errors="coerce")
df["CM_mean"]  = pd.to_numeric(df["Average Accuracy Contrastmedium"], errors="coerce")

# Lower and upper error bar for each metric
seq_xerr = np.vstack([df["Seq_mean"] - df["Seq_lower"],
                      df["Seq_upper"] - df["Seq_mean"]])
cm_xerr = np.vstack([df["CM_mean"] - df["CM_lower"],
                     df["CM_upper"] - df["CM_mean"]])

# Assign to a group
def classify(name: str) -> str:
    s = str(name).lower()
    if "radiolog" in s:
        return "Radiologists"
    if "rag" in s:
        return "with RAG"
    return "without RAG"

df["Group"] = df["Name"].apply(classify)

# Color palette sequences
seq_palette = {
    "without RAG":  "#B9AD91",
    "with RAG":     "#F75A3B",
    "Radiologists": "#4C72B0",
}

# Color palette contrast medium 
cm_palette = {
    "without RAG":  "#D1C29D",
    "with RAG":     "#F68E6F",
    "Radiologists": "#A9C3E6",
}

seq_colors = df["Group"].map(seq_palette).tolist()
cm_colors  = df["Group"].map(cm_palette).tolist()


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(19, 15), sharey=True)

y = np.arange(len(df))
h = 0.6

# Panel 1: MRI sequence prediction
ax1.barh(y, df["Seq_mean"], height=h, color=seq_colors, alpha=0.9, zorder=3)
ax1.errorbar(df["Seq_mean"], y, xerr=seq_xerr, fmt="none",
             ecolor="0.3", elinewidth=1.5, capsize=3, capthick=1.5, zorder=4)
ax1.set_title("MRI Sequence Prediction", fontweight="bold", pad=15, fontsize=25)
ax1.set_xlabel("Accuracy (%)")
ax1.set_xlim(0, 100)
ax1.set_yticks(y)
ax1.set_yticklabels(df["Name"])

# Panel 2: contrast medium prediction
ax2.barh(y, df["CM_mean"], height=h, color=cm_colors, alpha=0.9, zorder=3)
ax2.errorbar(df["CM_mean"], y, xerr=cm_xerr, fmt="none",
             ecolor="0.3", elinewidth=1.5, capsize=3, capthick=1.5, zorder=4)
ax2.set_title("Contrast Medium Prediction", fontweight="bold", pad=15, fontsize=25)
ax2.set_xlabel("Accuracy (%)")
ax2.set_xlim(0, 100)

sns.despine(left=True, bottom=True)

# Shared legend for both panels
legend_handles = [
    Patch(facecolor=seq_palette["without RAG"],  label="Models without RAG"),
    Patch(facecolor=seq_palette["with RAG"],     label="Models with RAG"),
    Patch(facecolor=seq_palette["Radiologists"], label="Radiologists"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=3, fontsize=20,
           bbox_to_anchor=(0.5, -0.05), frameon=True)

plt.tight_layout()
plt.show()

## MODEL COMPARISON WITH RAG

In [ ]:
# Load data 
df = pd.read_csv('/path/results',sep=';')
df = df.rename(columns={"Unnamed: 0": "Name"})

sns.set_theme(style="whitegrid", context="talk")

# Parse confidence interval 
def parse_ci(s):
    nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", str(s))
    return float(nums[0]), float(nums[1])

# Year of each model
if "Year" not in df.columns:
    df["Year"] = [2024, 2024, 2026, 2026, 2026]

# Split the interval strings into numeric bound columns
df[["Seq_lower", "Seq_upper"]] = df["Confidenz Interval Sequences"].apply(
    lambda x: pd.Series(parse_ci(x))
)
df[["CM_lower", "CM_upper"]] = df["Confidenz Interval Contrastmedium"].apply(
    lambda x: pd.Series(parse_ci(x))
)

# Convert to numeric values
df["Seq_mean"] = pd.to_numeric(df["Average Accuracy Sequences"], errors="coerce")
df["CM_mean"]  = pd.to_numeric(df["Average Accuracy Contrastmedium"], errors="coerce")

# Assign each model to a family 
def family_of(name: str) -> str:
    n = str(name).lower()
    if "llama" in n:
        return "Llama"
    if "gpt" in n:
        return "GPT"
    if "claude" in n:
        return "Claude"
    return "Other"

df["Family"] = df["Name"].apply(family_of)

# Color palette
family_style = {
    "Llama":  {"color": "#F68E6F", "marker": "o"},
    "GPT":    {"color": "#F75A3B", "marker": "s"},
    "Claude": {"color": "#BB462F", "marker": "D"},
}

# Two panels side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 10))
ax_seq, ax_cm = axes

metrics = [
    {"ax": ax_seq, "title": "MRI Sequence Prediction",
     "mean": "Seq_mean", "lower": "Seq_lower", "upper": "Seq_upper", "ylim": (60, 100)},
    {"ax": ax_cm, "title": "Contrast Medium Prediction",
     "mean": "CM_mean", "lower": "CM_lower", "upper": "CM_upper", "ylim": (60, 100)},
]

# Map each year to an x position
years = sorted(df["Year"].dropna().unique())
year_to_x = {yr: i for i, yr in enumerate(years)}
x_jitter = 0.04  # small horizontal offset so error bars do not overlap exactly

for m in metrics:
    ax = m["ax"]

    # One line per family, connecting its points across years
    for fam, style in family_style.items():
        sub = df[df["Family"] == fam].sort_values("Year").reset_index(drop=True)
        if sub.empty:
            continue

        # Tiny per family x offset to reduce overlap
        offset = {"Llama": -x_jitter, "GPT": 0.0, "Claude": x_jitter}[fam]
        xs = np.array([year_to_x[y] + offset for y in sub["Year"]])
        ys = sub[m["mean"]].to_numpy()
        yerr = np.vstack([ys - sub[m["lower"]].to_numpy(),
                          sub[m["upper"]].to_numpy() - ys])

        if len(sub) > 1:
            ax.plot(xs, ys, linestyle="-", color=style["color"],
                    linewidth=2, alpha=0.85, zorder=3)

        # Error bars and markers
        ax.errorbar(
            xs, ys, yerr=yerr,
            fmt=style["marker"], color=style["color"],
            ecolor=style["color"], elinewidth=1.5,
            capsize=4, capthick=1.5, markersize=10,
            markeredgecolor="white", markeredgewidth=0.8,
            zorder=4,
        )

        # Label each point with the model name
        for xi, yi, name in zip(xs, ys, sub["Name"]):
            ax.annotate(
                str(name),
                xy=(xi, yi),
                xytext=(25, -5),
                textcoords="offset points",
                fontsize=15,
                va="center",
                ha="left",
                color="0.15",
            )

    # Axis cosmetics
    ax.set_title(m["title"], fontweight="bold", fontsize=20, loc="left", pad=25)
    ax.set_ylim(*m["ylim"])
    ax.set_xticks(list(year_to_x.values()))
    ax.set_xticklabels([str(int(y)) for y in years])
    ax.set_xlim(-0.5, len(years) - 0.5)
    ax.set_ylabel("Accuracy (%)")
    ax.yaxis.set_major_locator(plt.MultipleLocator(5))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(2))
    ax.minorticks_on()
    ax.grid(True, axis="y", which="major", linewidth=0.8, alpha=0.9)
    ax.grid(True, axis="y", which="minor", linewidth=1, alpha=0.5)
    ax.grid(False, axis="x", which="both")
    sns.despine(ax=ax, left=False, bottom=False)

# Single legend at the bottom for families and the confidence interval
legend_handles = [
    plt.Line2D([0], [0], color=family_style["Llama"]["color"],
               marker="o", markersize=10, markeredgecolor="white",
               linestyle="-", linewidth=2, label="Llama"),
    plt.Line2D([0], [0], color=family_style["GPT"]["color"],
               marker="s", markersize=10, markeredgecolor="white",
               linestyle="-", linewidth=2, label="GPT"),
    plt.Line2D([0], [0], color=family_style["Claude"]["color"],
               marker="D", markersize=10, markeredgecolor="white",
               linestyle="-", linewidth=2, label="Claude"),
    plt.Line2D([0], [0], color="0.3", linestyle="-", linewidth=2, label="95% CI"),
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=4,
    frameon=False,
    fontsize=20,
)

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()